<a href="https://colab.research.google.com/github/kawastony/Quantum_Gravity/blob/main/Tests3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TAFA TRAJECTORY COMPARISON")
print("Best n=0 vs best n=1 vs CPL target")
print("="*60)

# ─── Cosmological parameters ────────────────────────────────
Omega_m0  = 0.300
Omega_r0  = 9.0e-5
Omega_DE0 = 1.0 - Omega_m0 - Omega_r0

# ─── Potential ───────────────────────────────────────────────
def V(rho, f, Lam4):
    return Lam4 * (1.0 - np.cos(rho / f))

def dV(rho, f, Lam4):
    return (Lam4 / f) * np.sin(rho / f)

# ─── Corrected angular momentum L = n*(f/2π)² ───────────────
def L_phys(n, f):
    return n * (f / (2.0 * np.pi))**2

def U_ang(rho, a, n, f):
    Lp = L_phys(n, f)
    return 0.5 * Lp**2 / (max(rho,1e-15)**2 * a**6)

def dU_ang_drho(rho, a, n, f):
    Lp = L_phys(n, f)
    return -Lp**2 / (max(rho,1e-15)**3 * a**6)

# ─── CPL curve ───────────────────────────────────────────────
def w_CPL(a, w0, wa):
    return w0 + wa * (1.0 - a)

# ─── CPL fit ─────────────────────────────────────────────────
def fit_CPL(a_arr, w_arr):
    mask = (a_arr >= 0.2) & (a_arr <= 1.0)
    if mask.sum() < 4:
        return None, None
    a_f = a_arr[mask]
    w_f = w_arr[mask]
    A   = np.column_stack([np.ones(len(a_f)), 1.0 - a_f])
    c, _, _, _ = np.linalg.lstsq(A, w_f, rcond=None)
    return float(c[0]), float(c[1])

# ─── ODE ─────────────────────────────────────────────────────
def make_rhs(f, Lam4, n):
    def rhs(N, Y):
        rho, y = Y
        a      = np.exp(N)
        rho    = max(rho, 1e-15)

        rho_r = 3.0 * Omega_r0  * np.exp(-4*N)
        rho_m = 3.0 * Omega_m0  * np.exp(-3*N)
        Vv    = V(rho, f, Lam4)
        Uv    = U_ang(rho, a, n, f)

        denom = max(1.0 - y**2/6.0, 1e-6)
        H2    = max((rho_r + rho_m + Uv + Vv)/(3.0*denom), 1e-30)

        KE      = 0.5 * H2 * y**2
        rho_phi = KE + Uv + Vv
        p_phi   = KE + Uv - Vv
        p_r     = rho_r / 3.0
        rho_tot = rho_r + rho_m + rho_phi
        p_tot   = p_r + p_phi
        w_tot   = p_tot/rho_tot if rho_tot > 1e-30 else -1.0

        HN_H  = -1.5*(1.0 + w_tot)
        dVdr  = dV(rho, f, Lam4)
        dUdr  = dU_ang_drho(rho, a, n, f)

        return [y, -(3.0 + HN_H)*y - (dVdr + dUdr)/H2]
    return rhs

# ─── Integration ─────────────────────────────────────────────
def run_TAFA(f, Lam4, n, rho_ini, y_ini=0.0,
             N_ini=-8.0, N_pts=4000):
    rhs  = make_rhs(f, Lam4, n)
    N_ev = np.linspace(N_ini, 0.0, N_pts)
    try:
        sol = solve_ivp(rhs, [N_ini, 0.0], [rho_ini, y_ini],
                        t_eval=N_ev, rtol=1e-10, atol=1e-13,
                        method='DOP853')
    except Exception:
        return None
    if sol.status != 0:
        return None

    a_arr   = np.exp(sol.t)
    rho_arr = sol.y[0]
    y_arr   = sol.y[1]
    w_arr   = np.zeros(N_pts)
    H2_arr  = np.zeros(N_pts)

    for i, N in enumerate(sol.t):
        a   = a_arr[i]
        rho = max(rho_arr[i], 1e-15)
        y   = y_arr[i]
        rho_r = 3.0*Omega_r0*np.exp(-4*N)
        rho_m = 3.0*Omega_m0*np.exp(-3*N)
        Vv  = V(rho, f, Lam4)
        Uv  = U_ang(rho, a, n, f)
        denom = max(1.0 - y**2/6.0, 1e-6)
        H2  = max((rho_r+rho_m+Uv+Vv)/(3.0*denom), 1e-30)
        H2_arr[i] = H2
        KE  = 0.5*H2*y**2
        r_p = KE+Uv+Vv
        p_p = KE+Uv-Vv
        w_arr[i] = p_p/r_p if r_p > 1e-10 else -1.0

    w0, wa = fit_CPL(a_arr, w_arr)
    rho0   = max(rho_arr[-1], 1e-15)
    Vv0    = V(rho0, f, Lam4)
    Uv0    = U_ang(rho0, 1.0, n, f)
    KE0    = 0.5*H2_arr[-1]*y_arr[-1]**2
    r_p0   = KE0+Uv0+Vv0
    Op0    = r_p0/(3.0*H2_arr[-1])

    return dict(a=a_arr, rho=rho_arr, y=y_arr,
                w=w_arr, H2=H2_arr,
                w0=w0, wa=wa,
                Omega_phi0=Op0,
                w0_direct=w_arr[-1])

# ═══════════════════════════════════════════════════════════
# STEP 1: Fine scan around the promising region
# Focus: small rho_ini/f ~ 0.005-0.05, n=0 and n=1
# ═══════════════════════════════════════════════════════════
print()
print("STEP 1: FINE SCAN — small ρ_ini/f region")
print("n=0 and n=1, f=0.05..0.50, ρ/f=0.001..0.05")
print()

w0_t = -0.827
wa_t = -0.750

f_fine    = [0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50]
Lam4_fine = np.logspace(-2, 2, 50)
rho_fine  = [0.001, 0.003, 0.005, 0.008, 0.010,
             0.015, 0.020, 0.030, 0.050, 0.080, 0.100]
n_fine    = [0, 1]

results_fine = []
count = 0

for fv in f_fine:
    for Lv in Lam4_fine:
        for rf in rho_fine:
            rho_ini = rf * fv
            if rho_ini < 1e-13:
                continue
            for n in n_fine:
                r = run_TAFA(fv, Lv, n, rho_ini, N_pts=3000)
                count += 1
                if r is None or r['w0'] is None:
                    continue
                if abs(r['Omega_phi0'] - Omega_DE0) > 0.20:
                    continue
                chi2 = (r['w0']-w0_t)**2 + (r['wa']-wa_t)**2
                results_fine.append((chi2, fv, Lv, n, rf,
                                     r['w0'], r['wa'],
                                     r['Omega_phi0'],
                                     r['a'].copy(),
                                     r['w'].copy()))

results_fine.sort(key=lambda x: x[0])
print(f"Evaluated {count} combinations, "
      f"{len(results_fine)} passed filter.\n")

print(f"{'χ²':>7}  {'f':>5}  {'Λ⁴':>8}  {'n':>3}  "
      f"{'ρ/f':>7}  {'w₀':>8}  {'wa':>8}  {'Ω_φ':>7}")
print("-"*65)
for row in results_fine[:15]:
    chi2,fv,Lv,n,rf,w0,wa,Op,_,_ = row
    print(f"{chi2:>7.4f}  {fv:>5.2f}  {Lv:>8.4f}  {n:>3}  "
          f"{rf:>7.4f}  {w0:>8.4f}  {wa:>8.4f}  {Op:>7.4f}")

# ─── Separate best n=0 and best n=1 ─────────────────────────
best_n0 = [x for x in results_fine if x[3]==0]
best_n1 = [x for x in results_fine if x[3]==1]

print()
print("─"*65)
if best_n0:
    c,fv,Lv,n,rf,w0,wa,Op,_,_ = best_n0[0]
    print(f"Best n=0:  χ²={c:.5f}  w₀={w0:.4f}  wa={wa:.4f}  "
          f"f={fv}  Λ⁴={Lv:.4f}  ρ/f={rf}")
if best_n1:
    c,fv,Lv,n,rf,w0,wa,Op,_,_ = best_n1[0]
    print(f"Best n=1:  χ²={c:.5f}  w₀={w0:.4f}  wa={wa:.4f}  "
          f"f={fv}  Λ⁴={Lv:.4f}  ρ/f={rf}")
print(f"Target:    w₀={w0_t}  wa={wa_t}")

# ═══════════════════════════════════════════════════════════
# STEP 2: w(a) trajectory comparison plot
# ═══════════════════════════════════════════════════════════
print()
print("STEP 2: TRAJECTORY COMPARISON")
print()

# Re-integrate best solutions at high resolution
def get_best_traj(best_list, N_pts=5000):
    if not best_list:
        return None
    c,fv,Lv,n,rf,w0,wa,Op,_,_ = best_list[0]
    rho_ini = rf * fv
    return run_TAFA(fv, Lv, n, rho_ini, N_pts=N_pts)

traj_n0 = get_best_traj(best_n0)
traj_n1 = get_best_traj(best_n1)

# CPL target
a_cpl = np.linspace(0.1, 1.0, 500)
w_cpl = w_CPL(a_cpl, w0_t, wa_t)

# Print trajectory table
print(f"{'a':>6}  {'z':>5}  "
      f"{'w_CPL':>8}  {'w_n0':>8}  {'w_n1':>8}  "
      f"{'Δ(n0)':>8}  {'Δ(n1)':>8}")
print("-"*60)

check_a = [0.20, 0.30, 0.40, 0.50, 0.60,
           0.70, 0.80, 0.90, 1.00]
for av in check_a:
    z_  = 1.0/av - 1.0
    wc  = w_CPL(av, w0_t, wa_t)

    if traj_n0 is not None:
        idx0 = np.argmin(np.abs(traj_n0['a'] - av))
        w0v  = traj_n0['w'][idx0]
        d0   = w0v - wc
    else:
        w0v = d0 = float('nan')

    if traj_n1 is not None:
        idx1 = np.argmin(np.abs(traj_n1['a'] - av))
        w1v  = traj_n1['w'][idx1]
        d1   = w1v - wc
    else:
        w1v = d1 = float('nan')

    print(f"{av:>6.2f}  {z_:>5.2f}  "
          f"{wc:>8.4f}  {w0v:>8.4f}  {w1v:>8.4f}  "
          f"{d0:>+8.4f}  {d1:>+8.4f}")

# ═══════════════════════════════════════════════════════════
# STEP 3: Plot w(a)
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: w(a) trajectories ──────────────────────────────
ax = axes[0]
ax.plot(a_cpl, w_cpl, 'k-', lw=2.5,
        label=f'CPL target  w₀={w0_t}, wa={wa_t}')
ax.axhline(-1.0, color='gray', ls=':', lw=1, alpha=0.6,
           label='w = −1 (cosmological constant)')

if traj_n0 is not None:
    mask = traj_n0['a'] >= 0.10
    c0,fv0,Lv0,n0,rf0,w0_0,wa_0,Op0,_,_ = best_n0[0]
    ax.plot(traj_n0['a'][mask], traj_n0['w'][mask],
            'b--', lw=2,
            label=f'n=0  w₀={w0_0:.3f} wa={wa_0:.3f}  χ²={c0:.4f}')

if traj_n1 is not None:
    mask = traj_n1['a'] >= 0.10
    c1,fv1,Lv1,n1,rf1,w0_1,wa_1,Op1,_,_ = best_n1[0]
    ax.plot(traj_n1['a'][mask], traj_n1['w'][mask],
            'r-', lw=2,
            label=f'n=1  w₀={w0_1:.3f} wa={wa_1:.3f}  χ²={c1:.4f}')

ax.set_xlabel('Scale factor  a', fontsize=12)
ax.set_ylabel('w(a)', fontsize=12)
ax.set_title('Equation of State Trajectory', fontsize=13)
ax.set_xlim(0.15, 1.05)
ax.set_ylim(-1.4, 0.2)
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)

# Mark today
ax.axvline(1.0, color='green', ls=':', lw=1, alpha=0.5)
ax.text(1.01, -1.35, 'today', fontsize=8, color='green')

# ── Right: w₀-wa plane ───────────────────────────────────
ax2 = axes[1]

# DESI 1σ and 2σ approximate contours (ellipses)
theta = np.linspace(0, 2*np.pi, 200)
# approximate DESI 1σ: σ(w0)~0.09, σ(wa)~0.33
for nsig, alpha, lw in [(1,0.35,2),(2,0.15,1)]:
    ax2.plot(w0_t + nsig*0.09*np.cos(theta),
             wa_t + nsig*0.33*np.sin(theta),
             'k-', lw=lw, alpha=alpha)
ax2.text(w0_t+0.02, wa_t+0.38,
         'DESI 1σ / 2σ\n(approx)', fontsize=8, ha='center')

ax2.plot(w0_t, wa_t, 'k*', ms=14, label='DESI target', zorder=5)

if best_n0:
    c0,fv0,Lv0,n0,rf0,w0_0,wa_0,Op0,_,_ = best_n0[0]
    ax2.plot(w0_0, wa_0, 'bs', ms=10,
             label=f'Best n=0  χ²={c0:.4f}', zorder=4)
    ax2.annotate(f'n=0\n({w0_0:.3f},{wa_0:.3f})',
                 xy=(w0_0,wa_0), xytext=(w0_0-0.12,wa_0+0.12),
                 fontsize=8, color='blue',
                 arrowprops=dict(arrowstyle='->', color='blue'))

if best_n1:
    c1,fv1,Lv1,n1,rf1,w0_1,wa_1,Op1,_,_ = best_n1[0]
    ax2.plot(w0_1, wa_1, 'r^', ms=10,
             label=f'Best n=1  χ²={c1:.4f}', zorder=4)
    ax2.annotate(f'n=1\n({w0_1:.3f},{wa_1:.3f})',
                 xy=(w0_1,wa_1), xytext=(w0_1+0.05,wa_1-0.15),
                 fontsize=8, color='red',
                 arrowprops=dict(arrowstyle='->', color='red'))

# Cosmological constant
ax2.plot(-1.0, 0.0, 'g^', ms=8, label='ΛCDM', zorder=3)

ax2.set_xlabel('w₀', fontsize=12)
ax2.set_ylabel('wa', fontsize=12)
ax2.set_title('w₀ – wa Plane', fontsize=13)
ax2.set_xlim(-1.3, -0.2)
ax2.set_ylim(-1.2, 0.5)
ax2.axhline(0, color='gray', ls=':', lw=0.8)
ax2.axvline(-1, color='gray', ls=':', lw=0.8)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tafa_trajectory_comparison.png',
            dpi=150, bbox_inches='tight')
print()
print("Plot saved: tafa_trajectory_comparison.png")

# ═══════════════════════════════════════════════════════════
# STEP 4: Honest diagnostic summary
# ═══════════════════════════════════════════════════════════
print()
print("="*60)
print("DIAGNOSTIC SUMMARY")
print("="*60)
print()
print(f"DESI target:   w₀ = {w0_t}   wa = {wa_t}")
print()

if best_n0 and best_n1:
    c0,_,_,_,_,w0_0,wa_0,_,_,_ = best_n0[0]
    c1,_,_,_,_,w0_1,wa_1,_,_,_ = best_n1[0]

    print(f"Best n=0:      w₀ = {w0_0:.4f}   wa = {wa_0:.4f}"
          f"   χ² = {c0:.4f}   dist = {np.sqrt(c0):.4f}")
    print(f"Best n=1:      w₀ = {w0_1:.4f}   wa = {wa_1:.4f}"
          f"   χ² = {c1:.4f}   dist = {np.sqrt(c1):.4f}")
    print()

    # Which is better
    if c1 < c0:
        impr = (c0-c1)/c0*100
        print(f"n=1 improves over n=0 by {impr:.1f}% in χ²")
    else:
        print("n=1 does NOT improve over n=0")
    print()

    # How far from target
    dist1 = np.sqrt(c1 if best_n1 else c0)
    best_c = min(c0, c1)
    dist_best = np.sqrt(best_c)

    print("Distance interpretation:")
    if dist_best < 0.10:
        verdict = "EXCELLENT — within observational scatter"
    elif dist_best < 0.20:
        verdict = "GOOD — close but not yet within 1σ"
    elif dist_best < 0.40:
        verdict = "PARTIAL — meaningful progress, not a match"
    else:
        verdict = "POOR — mechanism insufficient as-is"
    print(f"  √χ² = {dist_best:.4f}  →  {verdict}")
    print()

    # Remaining gaps
    best_row = best_n0[0] if c0 <= c1 else best_n1[0]
    w0_b, wa_b = best_row[5], best_row[6]
    print("Remaining gaps to close:")
    print(f"  Δw₀ = {w0_b - w0_t:+.4f}  "
          f"({'w₀ too high' if w0_b > w0_t else 'w₀ too low'})")
    print(f"  Δwa = {wa_b - wa_t:+.4f}  "
          f"({'wa too high' if wa_b > wa_t else 'wa too low'})")
    print()
    print("What these gaps mean physically:")
    if w0_b > w0_t:
        print("  → Field is not negative enough today.")
        print("    Potential needs steeper approach to minimum,")
        print("    or field needs to start further from minimum.")
    if wa_b > wa_t:
        print("  → EOS is not evolving fast enough.")
        print("    Transition needs to be sharper or later.")
    print()
    print("Next steps in order of priority:")
    print("  1. Examine full w(a) shape — is it the right")
    print("     qualitative form or just wrong everywhere?")
    print("  2. Try non-constant β(ρ) — this is the one")
    print("     unused geometric degree of freedom.")
    print("  3. Try different potential shapes beyond V_cos.")
    print("  4. If best dist > 0.30 after β(ρ) test,")
    print("     accept that this sector cannot reach DESI")
    print("     target and state that clearly.")

TAFA TRAJECTORY COMPARISON
Best n=0 vs best n=1 vs CPL target

STEP 1: FINE SCAN — small ρ_ini/f region
n=0 and n=1, f=0.05..0.50, ρ/f=0.001..0.05

Evaluated 7700 combinations, 1113 passed filter.

     χ²      f        Λ⁴    n      ρ/f        w₀        wa      Ω_φ
-----------------------------------------------------------------
 0.4102   0.20   15.2642    1   0.0500   -0.2096   -0.5795   0.8965
 0.4872   0.15    2.8118    1   0.1000   -0.1404   -0.8756   0.6575
 0.5010   0.15    2.3300    1   0.1000   -0.1257   -0.6543   0.5061
 0.5784   0.15    3.3932    1   0.1000   -0.0842   -0.9132   0.6235
 0.6234   0.30   22.2300    1   0.0500   -0.1223   -0.3940   0.8874
 0.6472   0.20   12.6486    1   0.0500   -0.0274   -0.6612   0.8133
 0.7575   0.20   10.4811    1   0.0500   -0.1047   -0.2645   0.7037
 0.7685   0.30    5.9636    1   0.0800   -0.0609   -0.3239   0.6547
 0.8292   0.30   18.4207    1   0.0500   -0.1685   -0.1210   0.8352
 0.8390   0.08   26.8270    1   0.0150   -0.0109   -0.33

In [ ]:

import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TAFA REGIME DIAGNOSTIC")
print("Enforce m_eff << H0 — find thawing solutions only")
print("="*60)

Omega_m0 = 0.300
Omega_r0 = 9.0e-5
Omega_DE0 = 1.0 - Omega_m0 - Omega_r0
H0 = 1.0  # units where H0=1

def V(rho, f, Lam4):
    return Lam4 * (1.0 - np.cos(rho / f))

def dV(rho, f, Lam4):
    return (Lam4 / f) * np.sin(rho / f)

def d2V(rho, f, Lam4):
    return (Lam4 / f**2) * np.cos(rho / f)

def L_phys(n, f):
    return n * (f / (2.0 * np.pi))**2

def U_ang(rho, a, n, f):
    Lp = L_phys(n, f)
    return 0.5 * Lp**2 / (max(rho, 1e-15)**2 * a**6)

def dU_ang_drho(rho, a, n, f):
    Lp = L_phys(n, f)
    return -Lp**2 / (max(rho, 1e-15)**3 * a**6)

def fit_CPL(a_arr, w_arr):
    mask = (a_arr >= 0.2) & (a_arr <= 1.0)
    if mask.sum() < 4:
        return None, None
    A = np.column_stack([np.ones(mask.sum()),
                         1.0 - a_arr[mask]])
    c, _, _, _ = np.linalg.lstsq(A, w_arr[mask], rcond=None)
    return float(c[0]), float(c[1])

def make_rhs(f, Lam4, n):
    def rhs(N, Y):
        rho, y = Y
        a = np.exp(N)
        rho = max(rho, 1e-15)
        rho_r = 3.0 * Omega_r0 * np.exp(-4*N)
        rho_m = 3.0 * Omega_m0 * np.exp(-3*N)
        Vv = V(rho, f, Lam4)
        Uv = U_ang(rho, a, n, f)
        denom = max(1.0 - y**2/6.0, 1e-6)
        H2 = max((rho_r + rho_m + Uv + Vv)/(3.0*denom), 1e-30)
        KE = 0.5 * H2 * y**2
        rho_phi = KE + Uv + Vv
        p_phi = KE + Uv - Vv
        p_r = rho_r / 3.0
        rho_tot = rho_r + rho_m + rho_phi
        p_tot = p_r + p_phi
        w_tot = p_tot/rho_tot if rho_tot > 1e-30 else -1.0
        HN_H = -1.5*(1.0 + w_tot)
        dVdr = dV(rho, f, Lam4)
        dUdr = dU_ang_drho(rho, a, n, f)
        return [y, -(3.0 + HN_H)*y - (dVdr + dUdr)/H2]
    return rhs

def run_TAFA(f, Lam4, n, rho_ini, N_ini=-8.0, N_pts=5000):
    rhs = make_rhs(f, Lam4, n)
    N_ev = np.linspace(N_ini, 0.0, N_pts)
    try:
        sol = solve_ivp(rhs, [N_ini, 0.0], [rho_ini, 0.0],
                        t_eval=N_ev, rtol=1e-10, atol=1e-13,
                        method='DOP853')
    except Exception:
        return None
    if sol.status != 0:
        return None

    a_arr = np.exp(sol.t)
    rho_arr = sol.y[0]
    y_arr = sol.y[1]
    w_arr = np.zeros(N_pts)
    H2_arr = np.zeros(N_pts)

    for i, N in enumerate(sol.t):
        a = a_arr[i]
        rho = max(rho_arr[i], 1e-15)
        y = y_arr[i]
        rho_r = 3.0*Omega_r0*np.exp(-4*N)
        rho_m = 3.0*Omega_m0*np.exp(-3*N)
        Vv = V(rho, f, Lam4)
        Uv = U_ang(rho, a, n, f)
        denom = max(1.0 - y**2/6.0, 1e-6)
        H2 = max((rho_r+rho_m+Uv+Vv)/(3.0*denom), 1e-30)
        H2_arr[i] = H2
        KE = 0.5*H2*y**2
        r_p = KE+Uv+Vv
        p_p = KE+Uv-Vv
        w_arr[i] = p_p/r_p if r_p > 1e-10 else -1.0

    w0, wa = fit_CPL(a_arr, w_arr)
    rho0 = max(rho_arr[-1], 1e-15)
    Vv0 = V(rho0, f, Lam4)
    Uv0 = U_ang(rho0, 1.0, n, f)
    KE0 = 0.5*H2_arr[-1]*y_arr[-1]**2
    r_p0 = KE0+Uv0+Vv0
    Op0 = r_p0/(3.0*H2_arr[-1])

    # Oscillation count: zero crossings of rho - rho_min
    rho_min = np.pi * f
    crossings = np.sum(np.diff(np.sign(rho_arr - rho_min)) != 0)

    # Effective mass at minimum vs H0
    m2_eff = d2V(rho_min, f, Lam4)
    m_eff = np.sqrt(max(m2_eff, 0))

    return dict(a=a_arr, rho=rho_arr, y=y_arr,
                w=w_arr, H2=H2_arr,
                w0=w0, wa=wa,
                Omega_phi0=Op0,
                crossings=crossings,
                m_eff=m_eff,
                rho_final=rho_arr[-1],
                rho_min=rho_min)

# ═══════════════════════════════════════════════════════════
# STEP 1: Mass regime scan
# For each (f, Lam4), compute m_eff and classify regime
# ═══════════════════════════════════════════════════════════
print()
print("STEP 1: MASS REGIME — which parameters give m_eff < H0?")
print(f"{'f':>6}  {'Λ⁴':>8}  {'m_eff':>10}  "
      f"{'m/H0':>8}  {'regime':>15}")
print("-"*55)

thawing_candidates = []

for fv in [0.1, 0.2, 0.3, 0.5, 1.0, 2.0]:
    for Lv in np.logspace(-6, 2, 20):
        rho_min_v = np.pi * fv
        m2 = (Lv / fv**2) * np.cos(0.0)  # at rho=0 slope
        # at minimum rho=pi*f: cos(pi)=-1 so V''=−Λ⁴/f²
        # Use rho slightly off min
        m2_min = Lv / fv**2  # magnitude at minimum
        m_eff = np.sqrt(m2_min)
        ratio = m_eff / H0
        if ratio < 0.5:
            regime = "THAWING candidate"
            thawing_candidates.append((fv, Lv))
        elif ratio < 5.0:
            regime = "borderline"
        else:
            regime = "oscillating"
        if ratio < 2.0:
            print(f"{fv:>6.1f}  {Lv:>8.2e}  {m_eff:>10.4e}  "
                  f"{ratio:>8.4f}  {regime:>15}")

# ═══════════════════════════════════════════════════════════
# STEP 2: Run thawing candidates, check oscillation count
# ═══════════════════════════════════════════════════════════
print()
print("="*60)
print("STEP 2: THAWING REGIME RUNS")
print("Only accept solutions with crossings < 3")
print()
print(f"{'f':>5}  {'Λ⁴':>8}  {'n':>3}  {'ρ_f/ρ_min':>10}  "
      f"{'cross':>6}  {'w₀':>8}  {'wa':>8}  {'Ω_φ':>7}  {'m/H0':>8}")
print("-"*70)

w0_t = -0.827
wa_t = -0.750
thawing_results = []

# Use only small Lam4 to keep m_eff < H0
f_thaw = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
Lam4_thaw = np.logspace(-6, -1, 30)
rho_fracs = [0.01, 0.05, 0.10, 0.20, 0.30,
             0.50, 0.70, 0.90, 0.95, 0.99]
n_vals = [0, 1]

for fv in f_thaw:
    for Lv in Lam4_thaw:
        m_eff = np.sqrt(Lv / fv**2)
        if m_eff > 3.0 * H0:
            continue  # skip oscillating regime
        rho_min_v = np.pi * fv
        for rf in rho_fracs:
            rho_ini = rf * rho_min_v
            for n in n_vals:
                r = run_TAFA(fv, Lv, n, rho_ini, N_pts=4000)
                if r is None or r['w0'] is None:
                    continue
                if r['crossings'] > 2:
                    continue  # oscillating, skip
                if abs(r['Omega_phi0'] - Omega_DE0) > 0.20:
                    continue
                chi2 = (r['w0']-w0_t)**2 + (r['wa']-wa_t)**2
                thawing_results.append(
                    (chi2, fv, Lv, n, rf,
                     r['w0'], r['wa'],
                     r['Omega_phi0'],
                     r['crossings'],
                     m_eff,
                     r['a'].copy(),
                     r['w'].copy(),
                     r['rho'].copy()))

thawing_results.sort(key=lambda x: x[0])

for row in thawing_results[:20]:
    chi2,fv,Lv,n,rf,w0,wa,Op,cross,meff,_,_,_ = row
    print(f"{fv:>5.1f}  {Lv:>8.2e}  {n:>3}  "
          f"{rf:>10.3f}  {cross:>6}  "
          f"{w0:>8.4f}  {wa:>8.4f}  {Op:>7.4f}  "
          f"{meff:>8.4f}")

# ═══════════════════════════════════════════════════════════
# STEP 3: Plot best thawing solution w(a)
# ═══════════════════════════════════════════════════════════
print()
print("STEP 3: TRAJECTORY PLOT — best thawing solution")

fig, axes = plt.subplots(2, 1, figsize=(10, 9))

a_cpl = np.linspace(0.1, 1.0, 300)
w_cpl = w0_t + wa_t*(1.0 - a_cpl)

ax = axes[0]
ax.plot(a_cpl, w_cpl, 'k-', lw=2.5,
        label=f'CPL target w₀={w0_t}, wa={wa_t}')
ax.axhline(-1.0, color='gray', ls=':', lw=1, alpha=0.5)

colors = ['red', 'blue', 'green']
for idx, row in enumerate(thawing_results[:3]):
    chi2,fv,Lv,n,rf,w0,wa,Op,cross,meff,a_arr,w_arr,_ = row
    mask = a_arr >= 0.10
    ax.plot(a_arr[mask], w_arr[mask],
            color=colors[idx], lw=1.8, alpha=0.8,
            label=f'f={fv} Λ⁴={Lv:.1e} n={n} '
                  f'cross={cross} '
                  f'w₀={w0:.3f} wa={wa:.3f}')

ax.set_xlabel('Scale factor a', fontsize=11)
ax.set_ylabel('w(a)', fontsize=11)
ax.set_title('w(a) — Thawing Regime Solutions Only', fontsize=12)
ax.set_xlim(0.1, 1.05)
ax.set_ylim(-1.3, 0.1)
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

# ─── Field trajectory ────────────────────────────────────
ax2 = axes[1]
for idx, row in enumerate(thawing_results[:3]):
    chi2,fv,Lv,n,rf,w0,wa,Op,cross,meff,a_arr,w_arr,rho_arr = row
    rho_min_v = np.pi * fv
    ax2.plot(a_arr, rho_arr / rho_min_v,
             color=colors[idx], lw=1.8, alpha=0.8,
             label=f'f={fv} ρ_ini/ρ_min={rf:.2f}')

ax2.axhline(1.0, color='black', ls='--', lw=1.5,
            label='Potential minimum (ρ=πf)')
ax2.set_xlabel('Scale factor a', fontsize=11)
ax2.set_ylabel('ρ / ρ_min', fontsize=11)
ax2.set_title('Field Trajectory — Does it stay near minimum?',
              fontsize=12)
ax2.set_xlim(0.0, 1.05)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tafa_thawing_regime.png', dpi=150,
            bbox_inches='tight')
print("Saved: tafa_thawing_regime.png")

# ═══════════════════════════════════════════════════════════
# STEP 4: Verdict
# ═══════════════════════════════════════════════════════════
print()
print("="*60)
print("VERDICT")
print("="*60)
print()
print(f"Target: w₀={w0_t}  wa={wa_t}")
print(f"Thawing solutions found: {len(thawing_results)}")
print()

if thawing_results:
    best = thawing_results[0]
    chi2,fv,Lv,n,rf,w0,wa,Op,cross,meff,_,_,_ = best
    dist = np.sqrt(chi2)
    print(f"Best thawing solution:")
    print(f"  f={fv}  Λ⁴={Lv:.3e}  n={n}")
    print(f"  ρ_ini = {rf:.2f} × ρ_min")
    print(f"  m_eff/H0 = {meff:.4f}")
    print(f"  oscillation crossings = {cross}")
    print(f"  w₀={w0:.4f}  wa={wa:.4f}")
    print(f"  Ω_φ={Op:.4f}")
    print(f"  distance = {dist:.4f}")
    print()
    if dist < 0.15:
        print("→ THAWING regime CAN reach DESI target.")
        print("  The oscillating regime was the problem.")
        print("  Enforce m_eff < H0 as a hard constraint.")
    elif dist < 0.35:
        print("→ Thawing regime improves significantly.")
        print("  Further refinement needed but mechanism viable.")
    else:
        print("→ Even in thawing regime, target not reached.")
        print("  Potential shape or field-space geometry")
        print("  needs modification beyond V_cos.")
else:
    print("→ No thawing solutions found with these parameters.")
    print("  Extend f and Λ⁴ ranges or reconsider potential.")

TAFA REGIME DIAGNOSTIC
Enforce m_eff << H0 — find thawing solutions only

STEP 1: MASS REGIME — which parameters give m_eff < H0?
     f        Λ⁴       m_eff      m/H0           regime
-------------------------------------------------------
   0.1  1.00e-06  1.0000e-02    0.0100  THAWING candidate
   0.1  2.64e-06  1.6238e-02    0.0162  THAWING candidate
   0.1  6.95e-06  2.6367e-02    0.0264  THAWING candidate
   0.1  1.83e-05  4.2813e-02    0.0428  THAWING candidate
   0.1  4.83e-05  6.9519e-02    0.0695  THAWING candidate
   0.1  1.27e-04  1.1288e-01    0.1129  THAWING candidate
   0.1  3.36e-04  1.8330e-01    0.1833  THAWING candidate
   0.1  8.86e-04  2.9764e-01    0.2976  THAWING candidate
   0.1  2.34e-03  4.8329e-01    0.4833  THAWING candidate
   0.1  6.16e-03  7.8476e-01    0.7848       borderline
   0.1  1.62e-02  1.2743e+00    1.2743       borderline
   0.2  1.00e-06  5.0000e-03    0.0050  THAWING candidate
   0.2  2.64e-06  8.1189e-03    0.0081  THAWING candidate
   0.2  

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TAFA — CORRECT THAWING WINDOW SCAN")
print("ρ_ini near top of potential slope, m/H0 ~ 0.3-1.0")
print("="*60)

Omega_m0 = 0.300
Omega_r0 = 9.0e-5
Omega_DE0 = 1.0 - Omega_m0 - Omega_r0

def V(rho, f, Lam4):
    x = rho / f
    return Lam4 * (1.0 - np.cos(x))

def dV(rho, f, Lam4):
    return (Lam4 / f) * np.sin(rho / f)

def d2V(rho, f, Lam4):
    return (Lam4 / f**2) * np.cos(rho / f)

def L_phys(n, f):
    return n * (f / (2.0 * np.pi))**2

def U_ang(rho, a, n, f):
    Lp = L_phys(n, f)
    return 0.5 * Lp**2 / (max(rho, 1e-15)**2 * a**6)

def dU_ang_drho(rho, a, n, f):
    Lp = L_phys(n, f)
    return -Lp**2 / (max(rho, 1e-15)**3 * a**6)

def fit_CPL(a_arr, w_arr):
    mask = (a_arr >= 0.2) & (a_arr <= 1.0)
    if mask.sum() < 4:
        return None, None
    A = np.column_stack([np.ones(mask.sum()),
                         1.0 - a_arr[mask]])
    c, _, _, _ = np.linalg.lstsq(A, w_arr[mask], rcond=None)
    return float(c[0]), float(c[1])

def make_rhs(f, Lam4, n):
    def rhs(N, Y):
        rho, y = Y
        a = np.exp(N)
        rho = max(rho, 1e-15)
        rho_r = 3.0 * Omega_r0 * np.exp(-4*N)
        rho_m = 3.0 * Omega_m0 * np.exp(-3*N)
        Vv  = V(rho, f, Lam4)
        Uv  = U_ang(rho, a, n, f)
        denom = max(1.0 - y**2/6.0, 1e-6)
        H2  = max((rho_r+rho_m+Uv+Vv)/(3.0*denom), 1e-30)
        KE  = 0.5*H2*y**2
        rho_phi = KE+Uv+Vv
        p_phi   = KE+Uv-Vv
        p_r     = rho_r/3.0
        rho_tot = rho_r+rho_m+rho_phi
        p_tot   = p_r+p_phi
        w_tot   = p_tot/rho_tot if rho_tot > 1e-30 else -1.0
        HN_H    = -1.5*(1.0+w_tot)
        dVdr    = dV(rho, f, Lam4)
        dUdr    = dU_ang_drho(rho, a, n, f)
        return [y, -(3.0+HN_H)*y - (dVdr+dUdr)/H2]
    return rhs

def run_TAFA(f, Lam4, n, rho_ini, N_ini=-8.0, N_pts=6000):
    rhs  = make_rhs(f, Lam4, n)
    N_ev = np.linspace(N_ini, 0.0, N_pts)
    try:
        sol = solve_ivp(rhs, [N_ini, 0.0], [rho_ini, 0.0],
                        t_eval=N_ev, rtol=1e-11, atol=1e-14,
                        method='DOP853')
    except Exception:
        return None
    if sol.status != 0:
        return None

    a_arr   = np.exp(sol.t)
    rho_arr = sol.y[0]
    y_arr   = sol.y[1]
    w_arr   = np.zeros(N_pts)
    H2_arr  = np.zeros(N_pts)

    for i, N in enumerate(sol.t):
        a   = a_arr[i]
        rho = max(rho_arr[i], 1e-15)
        y   = y_arr[i]
        rho_r = 3.0*Omega_r0*np.exp(-4*N)
        rho_m = 3.0*Omega_m0*np.exp(-3*N)
        Vv  = V(rho, f, Lam4)
        Uv  = U_ang(rho, a, n, f)
        denom = max(1.0 - y**2/6.0, 1e-6)
        H2  = max((rho_r+rho_m+Uv+Vv)/(3.0*denom), 1e-30)
        H2_arr[i] = H2
        KE  = 0.5*H2*y**2
        r_p = KE+Uv+Vv
        p_p = KE+Uv-Vv
        w_arr[i] = p_p/r_p if r_p > 1e-10 else -1.0

    # Oscillation crossing count
    rho_min = np.pi * f
    crossings = int(np.sum(
        np.diff(np.sign(rho_arr - rho_min)) != 0))

    w0, wa = fit_CPL(a_arr, w_arr)
    rho0 = max(rho_arr[-1], 1e-15)
    Vv0  = V(rho0, f, Lam4)
    Uv0  = U_ang(rho0, 1.0, n, f)
    KE0  = 0.5*H2_arr[-1]*y_arr[-1]**2
    r_p0 = KE0+Uv0+Vv0
    Op0  = r_p0/(3.0*H2_arr[-1])
    m_eff = np.sqrt(max(Lam4/f**2, 0))

    return dict(a=a_arr, rho=rho_arr, y=y_arr,
                w=w_arr, H2=H2_arr,
                w0=w0, wa=wa,
                Omega_phi0=Op0,
                crossings=crossings,
                m_eff=m_eff,
                rho_final=rho_arr[-1],
                rho_min=rho_min,
                w_today=w_arr[-1],
                V_today=V(rho_arr[-1], f, Lam4),
                KE_today=0.5*H2_arr[-1]*y_arr[-1]**2)

# ═══════════════════════════════════════════════════════════
# SCAN: field starts high on slope (near potential maximum)
# This is the correct thawing initial condition
# rho_ini should be LARGE (near or above pi*f)
# but field should not have crossed minimum yet
# ═══════════════════════════════════════════════════════════

w0_t = -0.827
wa_t = -0.750

print()
print("Scanning: high ρ_ini, moderate m/H0")
print()
print(f"{'f':>4}  {'Λ⁴':>8}  {'n':>2}  "
      f"{'ρ_ini/πf':>9}  {'cross':>5}  "
      f"{'w₀':>8}  {'wa':>8}  {'Ω_φ':>6}  "
      f"{'m/H0':>6}  {'χ²':>8}")
print("-"*80)

# Key insight: for thawing, start near potential MAXIMUM
# V_cos has maximum at rho = 0 (if we use 1-cos)
# Actually maximum of V = Lam4*(1-cos(rho/f)) is at rho = pi*f
# So field starts BEFORE the maximum: rho_ini < pi*f
# but close enough to pi*f that V is large (near 2*Lam4)
# and field has barely started rolling

f_scan    = [0.3, 0.5, 1.0, 1.5, 2.0, 3.0]
Lam4_scan = np.logspace(-3, 1, 25)
# rho_ini as fraction of rho_max = pi*f
# Start near maximum of V, not near minimum
rho_fracs = [0.30, 0.40, 0.50, 0.60, 0.70,
             0.75, 0.80, 0.85, 0.90, 0.95, 0.98]
n_vals = [0, 1]

results = []
count   = 0

for fv in f_scan:
    rho_max = np.pi * fv  # potential maximum
    for Lv in Lam4_scan:
        m_eff = np.sqrt(Lv / fv**2)
        for rf in rho_fracs:
            rho_ini = rf * rho_max
            for n in n_vals:
                r = run_TAFA(fv, Lv, n, rho_ini, N_pts=5000)
                count += 1
                if r is None or r['w0'] is None:
                    continue
                if r['crossings'] > 1:
                    continue  # must not oscillate
                if abs(r['Omega_phi0'] - Omega_DE0) > 0.25:
                    continue
                chi2 = (r['w0']-w0_t)**2 + (r['wa']-wa_t)**2
                results.append((
                    chi2, fv, Lv, n, rf,
                    r['w0'], r['wa'],
                    r['Omega_phi0'],
                    r['crossings'],
                    m_eff,
                    r['a'].copy(),
                    r['w'].copy(),
                    r['rho'].copy(),
                    r['w_today'],
                    r['V_today'],
                    r['KE_today']))

results.sort(key=lambda x: x[0])

print(f"Evaluated {count} combinations, "
      f"{len(results)} passed filters.\n")

for row in results[:25]:
    (chi2,fv,Lv,n,rf,w0,wa,Op,
     cross,meff,_,_,_,wt,Vt,KEt) = row
    print(f"{fv:>4.1f}  {Lv:>8.2e}  {n:>2}  "
          f"{rf:>9.3f}  {cross:>5}  "
          f"{w0:>8.4f}  {wa:>8.4f}  {Op:>6.4f}  "
          f"{meff:>6.4f}  {chi2:>8.4f}")

# ═══════════════════════════════════════════════════════════
# w(a) SHAPE DIAGNOSTIC for top 3 solutions
# ═══════════════════════════════════════════════════════════
print()
print("w(a) shape at key redshifts — top 3 solutions:")
print()
print(f"{'a':>5}  {'z':>4}  {'w_CPL':>7}  ", end='')
for i in range(min(3, len(results))):
    print(f"{'soln'+str(i+1):>8}  ", end='')
print()
print("-"*50)

a_check = [0.20, 0.33, 0.50, 0.67, 0.80, 0.91, 1.00]
for av in a_check:
    zv  = 1.0/av - 1.0
    wc  = w0_t + wa_t*(1.0-av)
    print(f"{av:>5.2f}  {zv:>4.2f}  {wc:>7.4f}  ", end='')
    for i in range(min(3, len(results))):
        _,_,_,_,_,_,_,_,_,_,a_arr,w_arr,_,_,_,_ = results[i]
        idx = np.argmin(np.abs(a_arr - av))
        print(f"{w_arr[idx]:>8.4f}  ", end='')
    print()

# ═══════════════════════════════════════════════════════════
# PLOT
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

a_cpl = np.linspace(0.1, 1.0, 300)
w_cpl = w0_t + wa_t*(1.0 - a_cpl)

# ── Left: w(a) ───────────────────────────────────────────
ax = axes[0]
ax.plot(a_cpl, w_cpl, 'k-', lw=2.5,
        label=f'CPL target')
ax.axhline(-1.0, color='gray', ls=':', lw=1, alpha=0.5)

colors = ['red','blue','green','orange','purple']
for i, row in enumerate(results[:5]):
    (chi2,fv,Lv,n,rf,w0,wa,Op,
     cross,meff,a_arr,w_arr,_,_,_,_) = row
    mask = a_arr >= 0.10
    ax.plot(a_arr[mask], w_arr[mask],
            color=colors[i], lw=1.5, alpha=0.8,
            label=f'f={fv} n={n} χ²={chi2:.3f}')

ax.set_xlabel('a', fontsize=11)
ax.set_ylabel('w(a)', fontsize=11)
ax.set_title('w(a) Trajectories', fontsize=12)
ax.set_xlim(0.1, 1.05)
ax.set_ylim(-1.3, 0.3)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# ── Middle: w₀-wa plane ──────────────────────────────────
ax2 = axes[1]
theta = np.linspace(0, 2*np.pi, 200)
for ns, al in [(1,0.35),(2,0.15)]:
    ax2.plot(w0_t+ns*0.09*np.cos(theta),
             wa_t+ns*0.33*np.sin(theta),
             'k-', lw=1.5, alpha=al)

ax2.plot(w0_t, wa_t, 'k*', ms=14,
         label='DESI target', zorder=5)
ax2.plot(-1.0, 0.0, 'g^', ms=8,
         label='ΛCDM', zorder=4)

for i, row in enumerate(results[:10]):
    chi2,fv,Lv,n,rf,w0,wa = row[:7]
    ax2.scatter(w0, wa, color=colors[min(i,4)],
                s=60, zorder=3, alpha=0.7)

if results:
    w0b, wab = results[0][5], results[0][6]
    ax2.annotate(f'best\n({w0b:.3f},{wab:.3f})',
                 xy=(w0b,wab),
                 xytext=(w0b-0.1, wab-0.2),
                 fontsize=8, color='red',
                 arrowprops=dict(arrowstyle='->',color='red'))

ax2.set_xlabel('w₀', fontsize=11)
ax2.set_ylabel('wa', fontsize=11)
ax2.set_title('w₀–wa Plane', fontsize=12)
ax2.set_xlim(-1.4, 0.5)
ax2.set_ylim(-1.5, 0.8)
ax2.axhline(0, color='gray', ls=':', lw=0.8)
ax2.axvline(-1, color='gray', ls=':', lw=0.8)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── Right: field trajectory ───────────────────────────────
ax3 = axes[2]
for i, row in enumerate(results[:3]):
    (chi2,fv,Lv,n,rf,w0,wa,Op,
     cross,meff,a_arr,w_arr,rho_arr,_,_,_) = row
    rho_max_v = np.pi * fv
    ax3.plot(a_arr, rho_arr/rho_max_v,
             color=colors[i], lw=1.5,
             label=f'f={fv} χ²={chi2:.3f}')

ax3.axhline(1.0, color='black', ls='--', lw=1.5,
            label='ρ = ρ_max (V peak)')
ax3.axhline(0.0, color='gray', ls=':', lw=1)
ax3.set_xlabel('a', fontsize=11)
ax3.set_ylabel('ρ / ρ_max', fontsize=11)
ax3.set_title('Field Position (normalized)', fontsize=12)
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tafa_correct_thawing.png', dpi=150,
            bbox_inches='tight')
print()
print("Saved: tafa_correct_thawing.png")

# ═══════════════════════════════════════════════════════════
# HONEST SUMMARY
# ═══════════════════════════════════════════════════════════
print()
print("="*60)
print("HONEST SUMMARY")
print("="*60)
print()
print(f"Target: w₀={w0_t}  wa={wa_t}")
print()

if results:
    best = results[0]
    chi2,fv,Lv,n,rf,w0,wa,Op = best[:8]
    dist = np.sqrt(chi2)
    print(f"Best result:  w₀={w0:.4f}  wa={wa:.4f}")
    print(f"Distance:     {dist:.4f}")
    print(f"Parameters:   f={fv}  Λ⁴={Lv:.3e}  n={n}"
          f"  ρ_ini={rf:.2f}×πf")
    print()

    # Track progress
    print("Progress history (approximate):")
    print(f"  Coarse scan:      dist ≈ 0.55")
    print(f"  Corrected L:      dist ≈ 0.35")
    print(f"  High-res scan:    dist ≈ 0.21")
    print(f"  Oscillating avg:  dist ≈ 0.46 (w₀=−0.37)")
    print(f"  Wrong thawing:    dist ≈ 1.94 (w₀=+0.87)")
    print(f"  This scan:        dist ≈ {dist:.2f}")
    print()

    if dist < 0.15:
        print("STATUS: CLOSE — within observational range")
        print("The correct initial condition regime found.")
    elif dist < 0.35:
        print("STATUS: IMPROVING — right direction")
        print("Refine parameters further.")
    else:
        print("STATUS: V_cos alone cannot reach the target.")
        print()
        print("The fundamental issue:")
        print("  V = Λ⁴(1 - cos(ρ/f)) has a single free")
        print("  shape — it cannot independently control")
        print("  both w₀ and wa. The ratio wa/w₀ is")
        print("  constrained by the cosine shape itself.")
        print()
        print("What would actually help:")
        print("  1. Two-term potential: V = Λ⁴(1-cos(ρ/f))")
        print("                            + αΛ⁴cos(2ρ/f)")
        print("     This adds one shape parameter.")
        print("  2. Non-trivial β(ρ) from Appendix A.1")
        print("     This modifies the effective potential")
        print("     through the field-space curvature.")
        print("  3. Accept that this model cannot match")
        print("     the DESI target and state that clearly.")
else:
    print("No valid thawing solutions found.")
    print("The V_cos potential in standard slow-roll")
    print("does not produce dark energy in this range.")

TAFA — CORRECT THAWING WINDOW SCAN
ρ_ini near top of potential slope, m/H0 ~ 0.3-1.0

Scanning: high ρ_ini, moderate m/H0

   f        Λ⁴   n   ρ_ini/πf  cross        w₀        wa     Ω_φ    m/H0        χ²
--------------------------------------------------------------------------------
Evaluated 3300 combinations, 705 passed filters.

 0.3  2.15e+00   1      0.980      1   -0.6096   -0.5799  0.7485  4.8927    0.0762
 0.5  4.64e+00   0      0.800      0   -0.5806   -0.5999  0.8403  4.3089    0.0832
 0.5  2.15e+00   0      0.700      0   -0.5673   -0.6194  0.6782  2.9356    0.0845
 0.3  4.64e+00   1      0.300      1   -0.6399   -0.5240  0.8713  7.1814    0.0861
 0.5  3.16e+00   0      0.750      0   -0.5521   -0.6413  0.7652  3.5566    0.0874
 0.3  4.64e+00   0      0.950      0   -0.5223   -0.7307  0.8576  7.1814    0.0932
 0.3  1.00e+01   0      0.980      0   -0.6917   -0.4741  0.9406  10.5409    0.0944
 0.3  6.81e+00   1      0.300      1   -0.6771   -0.4796  0.9130  8.7005    0.095

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TAFA — LATE-TIME ROLL SPEED REFINEMENT")
print("Goal: field must still be near w=-0.83 at a=1")
print("="*60)

Omega_m0  = 0.300
Omega_r0  = 9.0e-5
Omega_DE0 = 1.0 - Omega_m0 - Omega_r0

# ── Potential ────────────────────────────────────────────
def V(rho, f, Lam4):
    return Lam4 * (1.0 - np.cos(rho / f))

def dV(rho, f, Lam4):
    return (Lam4 / f) * np.sin(rho / f)

# ── Angular momentum (n=0 or n=1) ───────────────────────
def U_ang(rho, a, n, f, beta):
    if n == 0:
        return 0.0
    Lp = n * (f / (2.0 * np.pi))**2
    return 0.5 * Lp**2 / (max(beta*rho, 1e-15)**2
                          / beta**2 * a**6)

def dU_drho(rho, a, n, f, beta):
    if n == 0:
        return 0.0
    Lp = n * (f / (2.0 * np.pi))**2
    return -Lp**2 / (max(beta**2 * rho**3, 1e-45) * a**6)

# ── RHS with explicit β ──────────────────────────────────
def make_rhs(f, Lam4, n, beta):
    def rhs(N, Y):
        rho, y = Y
        a   = np.exp(N)
        rho = max(rho, 1e-15)
        rr  = 3.0 * Omega_r0  * np.exp(-4*N)
        rm  = 3.0 * Omega_m0  * np.exp(-3*N)
        Vv  = V(rho, f, Lam4)
        Uv  = U_ang(rho, a, n, f, beta)
        den = max(1.0 - y**2/6.0, 1e-6)
        H2  = max((rr+rm+Uv+Vv)/(3.0*den), 1e-30)
        KE  = 0.5*H2*y**2
        r_p = KE+Uv+Vv
        p_p = KE+Uv-Vv
        w_t = (rr/3.0 + p_p) / max(rr+rm+r_p, 1e-30)
        HH  = -1.5*(1.0 + w_t)
        force = dV(rho, f, Lam4) + dU_drho(rho, a, n, f, beta)
        return [y, -(3.0+HH)*y - force/H2]
    return rhs

def fit_CPL(a_arr, w_arr):
    mask = (a_arr >= 0.2) & (a_arr <= 1.0)
    if mask.sum() < 4:
        return None, None
    A  = np.column_stack([np.ones(mask.sum()),
                          1.0 - a_arr[mask]])
    c,_,_,_ = np.linalg.lstsq(A, w_arr[mask], rcond=None)
    return float(c[0]), float(c[1])

def run(f, Lam4, n, beta, rho_ini,
        N_ini=-8.0, N_pts=6000):
    rhs  = make_rhs(f, Lam4, n, beta)
    N_ev = np.linspace(N_ini, 0.0, N_pts)
    try:
        sol = solve_ivp(rhs, [N_ini, 0.0],
                        [rho_ini, 0.0],
                        t_eval=N_ev,
                        rtol=1e-11, atol=1e-14,
                        method='DOP853')
    except Exception:
        return None
    if sol.status != 0:
        return None

    a_arr   = np.exp(sol.t)
    rho_arr = sol.y[0]
    y_arr   = sol.y[1]
    w_arr   = np.zeros(N_pts)
    H2_arr  = np.zeros(N_pts)

    for i, N in enumerate(sol.t):
        a   = a_arr[i]
        rho = max(rho_arr[i], 1e-15)
        y   = y_arr[i]
        rr  = 3.0*Omega_r0 *np.exp(-4*N)
        rm  = 3.0*Omega_m0 *np.exp(-3*N)
        Vv  = V(rho, f, Lam4)
        Uv  = U_ang(rho, a, n, f, beta)
        den = max(1.0 - y**2/6.0, 1e-6)
        H2  = max((rr+rm+Uv+Vv)/(3.0*den), 1e-30)
        H2_arr[i] = H2
        KE  = 0.5*H2*y**2
        r_p = KE+Uv+Vv
        p_p = KE+Uv-Vv
        w_arr[i] = p_p/r_p if r_p > 1e-10 else -1.0

    rho_max_v = np.pi * f
    cross = int(np.sum(
        np.diff(np.sign(rho_arr - rho_max_v)) != 0))

    w0, wa = fit_CPL(a_arr, w_arr)

    rho0 = max(rho_arr[-1], 1e-15)
    Vv0  = V(rho0, f, Lam4)
    Uv0  = U_ang(rho0, 1.0, n, f, beta)
    KE0  = 0.5*H2_arr[-1]*y_arr[-1]**2
    r_p0 = KE0+Uv0+Vv0
    Op0  = r_p0/(3.0*H2_arr[-1])

    return dict(a=a_arr, rho=rho_arr, w=w_arr,
                H2=H2_arr,
                w0=w0, wa=wa,
                Omega_phi0=Op0,
                crossings=cross,
                w_today=w_arr[-1],
                w_at={av: w_arr[np.argmin(np.abs(a_arr-av))]
                      for av in [0.5, 0.67, 0.8, 0.91, 1.0]})

# ═══════════════════════════════════════════════════════════
# KEY NEW LEVER: β < 1 slows the radial roll
# Scan β from 1.0 down to 0.1
# ═══════════════════════════════════════════════════════════

w0_t = -0.827
wa_t = -0.750

print()
print("PRIMARY SCAN: β(ρ) as geometric brake")
print()
print(f"{'f':>4}  {'Λ⁴':>7}  {'n':>2}  {'β':>5}  "
      f"{'ρ_ini/πf':>9}  {'w_today':>8}  "
      f"{'w₀':>8}  {'wa':>8}  {'Ω_φ':>6}  {'χ²':>8}")
print("-"*80)

# Refined ranges based on previous best
f_scan    = [0.3, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
Lam4_scan = np.logspace(-1, 2, 20)
beta_scan = [1.0, 0.8, 0.6, 0.5, 0.4, 0.3,
             0.25, 0.2, 0.15, 0.10]
rho_fracs = [0.70, 0.75, 0.80, 0.85,
             0.90, 0.93, 0.96, 0.98]
n_vals    = [0, 1]

results = []

for fv in f_scan:
    rho_max = np.pi * fv
    for Lv in Lam4_scan:
        for beta in beta_scan:
            for rf in rho_fracs:
                rho_ini = rf * rho_max
                for n in n_vals:
                    r = run(fv, Lv, n, beta, rho_ini,
                            N_pts=5000)
                    if r is None or r['w0'] is None:
                        continue
                    if r['crossings'] > 1:
                        continue
                    if abs(r['Omega_phi0']-Omega_DE0) > 0.25:
                        continue
                    # KEY NEW FILTER: w today must be near -0.83
                    if r['w_today'] > -0.50:
                        continue
                    chi2 = ((r['w0']-w0_t)**2 +
                            (r['wa']-wa_t)**2)
                    results.append((
                        chi2, fv, Lv, n, beta, rf,
                        r['w0'], r['wa'],
                        r['Omega_phi0'],
                        r['crossings'],
                        r['w_today'],
                        r['w_at'],
                        r['a'].copy(),
                        r['w'].copy(),
                        r['rho'].copy()))

results.sort(key=lambda x: x[0])
print(f"Valid solutions found: {len(results)}\n")

for row in results[:20]:
    (chi2,fv,Lv,n,beta,rf,
     w0,wa,Op,cross,wt,_,_,_,_) = row
    print(f"{fv:>4.1f}  {Lv:>7.2e}  {n:>2}  "
          f"{beta:>5.2f}  {rf:>9.3f}  "
          f"{wt:>8.4f}  "
          f"{w0:>8.4f}  {wa:>8.4f}  "
          f"{Op:>6.4f}  {chi2:>8.4f}")

# ═══════════════════════════════════════════════════════════
# SHAPE TABLE
# ═══════════════════════════════════════════════════════════
print()
print("w(a) shape — top 3 vs CPL target:")
print()
hdr = f"{'a':>5}  {'z':>4}  {'w_CPL':>7}  "
for i in range(min(3, len(results))):
    hdr += f"{'sol'+str(i+1):>8}  "
print(hdr)
print("-"*55)

for av, zv in [(0.20,4.00),(0.33,2.03),(0.50,1.00),
               (0.67,0.49),(0.80,0.25),(0.91,0.10),
               (1.00,0.00)]:
    wc  = w0_t + wa_t*(1.0-av)
    line = f"{av:>5.2f}  {zv:>4.2f}  {wc:>7.4f}  "
    for i in range(min(3, len(results))):
        wa_at = results[i][11]
        wv = wa_at.get(av,
             results[i][13][
                 np.argmin(np.abs(results[i][12]-av))])
        line += f"{wv:>8.4f}  "
    print(line)

# ═══════════════════════════════════════════════════════════
# PLOT
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
a_cpl = np.linspace(0.1, 1.0, 300)
w_cpl = w0_t + wa_t*(1.0 - a_cpl)
colors = ['red','blue','green','orange','purple']

# w(a)
ax = axes[0]
ax.plot(a_cpl, w_cpl, 'k-', lw=2.5, label='CPL target')
ax.axhline(-1.0, color='gray', ls=':', lw=1)
for i, row in enumerate(results[:5]):
    chi2,fv,Lv,n,beta,rf,w0,wa = row[:8]
    a_arr, w_arr = row[12], row[13]
    mask = a_arr >= 0.10
    ax.plot(a_arr[mask], w_arr[mask],
            color=colors[i], lw=1.5, alpha=0.85,
            label=f'f={fv} β={beta} χ²={chi2:.3f}')
ax.set_xlabel('a'); ax.set_ylabel('w(a)')
ax.set_title('w(a) — with β geometric brake')
ax.set_xlim(0.1, 1.05); ax.set_ylim(-1.3, 0.1)
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# w0-wa plane
ax2 = axes[1]
theta = np.linspace(0, 2*np.pi, 200)
for ns, al in [(1,0.4),(2,0.2),(3,0.1)]:
    ax2.plot(w0_t + ns*0.09*np.cos(theta),
             wa_t + ns*0.33*np.sin(theta),
             'k-', lw=1.2, alpha=al)
ax2.plot(w0_t, wa_t, 'k*', ms=14,
         label='DESI target', zorder=5)
ax2.plot(-1.0, 0.0, 'g^', ms=8, label='ΛCDM')
for i, row in enumerate(results[:15]):
    chi2,fv,Lv,n,beta,rf,w0,wa = row[:8]
    ax2.scatter(w0, wa, color=colors[min(i,4)],
                s=50, zorder=3, alpha=0.7)
if results:
    w0b,wab = results[0][6], results[0][7]
    ax2.annotate(f'best ({w0b:.3f},{wab:.3f})',
                 xy=(w0b,wab),
                 xytext=(w0b+0.05, wab-0.2),
                 fontsize=8, color='red',
                 arrowprops=dict(arrowstyle='->',
                                 color='red'))
ax2.set_xlabel('w₀'); ax2.set_ylabel('wa')
ax2.set_title('w₀–wa Plane')
ax2.set_xlim(-1.3, -0.3); ax2.set_ylim(-1.5, 0.5)
ax2.axhline(0,color='gray',ls=':',lw=0.8)
ax2.axvline(-1,color='gray',ls=':',lw=0.8)
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# β effect: distance vs β for best f
ax3 = axes[2]
beta_vals = sorted(set(r[4] for r in results))
for fv in [0.3, 0.5, 1.0]:
    bv_list, dist_list = [], []
    for beta in beta_vals:
        sub = [r for r in results
               if abs(r[1]-fv)<0.01
               and abs(r[4]-beta)<0.01]
        if sub:
            best_sub = min(sub, key=lambda x: x[0])
            bv_list.append(beta)
            dist_list.append(np.sqrt(best_sub[0]))
    if bv_list:
        ax3.plot(bv_list, dist_list, 'o-',
                 label=f'f={fv}', lw=1.5)
ax3.axhline(0.15, color='red', ls='--', lw=1.5,
            label='1σ threshold')
ax3.axhline(0.09, color='orange', ls='--', lw=1.5,
            label='target zone')
ax3.set_xlabel('β (field-space curvature)')
ax3.set_ylabel('Distance to DESI target')
ax3.set_title('β as Geometric Brake')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)
ax3.invert_xaxis()

plt.tight_layout()
plt.savefig('tafa_beta_refinement.png', dpi=150,
            bbox_inches='tight')
print("\nSaved: tafa_beta_refinement.png")

# ═══════════════════════════════════════════════════════════
# VERDICT
# ═══════════════════════════════════════════════════════════
print()
print("="*60)
print("VERDICT")
print("="*60)
if results:
    best = results[0]
    chi2,fv,Lv,n,beta,rf,w0,wa,Op = best[:9]
    dist = np.sqrt(chi2)
    print(f"\nBest: w₀={w0:.4f}  wa={wa:.4f}"
          f"  dist={dist:.4f}")
    print(f"Params: f={fv}  Λ⁴={Lv:.3e}"
          f"  n={n}  β={beta}  ρ_ini={rf:.2f}×πf")
    print()
    if dist < 0.09:
        print("WITHIN 1σ OF DESI TARGET.")
        print("Document β as a physically motivated")
        print("parameter from Appendix A.1 geometry.")
    elif dist < 0.20:
        print("Close. β is working as geometric brake.")
        print("Fine-tune β and ρ_ini with denser grid.")
    else:
        print("β alone insufficient.")
        print("Consider two-term potential or")
        print("ρ-dependent β(ρ) = β₀·exp(-ρ/ρ_c)")

TAFA — LATE-TIME ROLL SPEED REFINEMENT
Goal: field must still be near w=-0.83 at a=1

PRIMARY SCAN: β(ρ) as geometric brake

   f       Λ⁴   n      β   ρ_ini/πf   w_today        w₀        wa     Ω_φ        χ²
--------------------------------------------------------------------------------
Valid solutions found: 5869

 0.5  3.79e+00   0   1.00      0.800   -0.5200   -0.6621   -0.4825  0.8274    0.0988
 0.5  3.79e+00   0   0.80      0.800   -0.5200   -0.6621   -0.4825  0.8274    0.0988
 0.5  3.79e+00   0   0.60      0.800   -0.5200   -0.6621   -0.4825  0.8274    0.0988
 0.5  3.79e+00   0   0.50      0.800   -0.5200   -0.6621   -0.4825  0.8274    0.0988
 0.5  3.79e+00   0   0.40      0.800   -0.5200   -0.6621   -0.4825  0.8274    0.0988
 0.5  3.79e+00   0   0.30      0.800   -0.5200   -0.6621   -0.4825  0.8274    0.0988
 0.5  3.79e+00   0   0.25      0.800   -0.5200   -0.6621   -0.4825  0.8274    0.0988
 0.5  3.79e+00   0   0.20      0.800   -0.5200   -0.6621   -0.4825  0.8274    0.0988
 

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TAFA — HONEST TRAJECTORY MATCHING")
print("Match actual w(a) shape, not just CPL fit")
print("="*60)

Omega_m0  = 0.300
Omega_r0  = 9.0e-5
Omega_DE0 = 1.0 - Omega_m0 - Omega_r0

def V(rho, f, Lam4, alpha=0.0):
    """Two-term potential: standard + correction"""
    x = rho / f
    return Lam4 * (1.0 - np.cos(x) + alpha*(1.0 - np.cos(2*x)))

def dV(rho, f, Lam4, alpha=0.0):
    x = rho / f
    return (Lam4/f) * (np.sin(x) + 2.0*alpha*np.sin(2*x))

def make_rhs(f, Lam4, alpha):
    def rhs(N, Y):
        rho, y = Y
        a   = np.exp(N)
        rho = max(rho, 1e-15)
        rr  = 3.0*Omega_r0*np.exp(-4*N)
        rm  = 3.0*Omega_m0*np.exp(-3*N)
        Vv  = V(rho, f, Lam4, alpha)
        den = max(1.0 - y**2/6.0, 1e-6)
        H2  = max((rr+rm+Vv)/(3.0*den), 1e-30)
        KE  = 0.5*H2*y**2
        r_p = KE + Vv
        p_p = KE - Vv
        w_t = (rr/3.0 + p_p)/max(rr+rm+r_p, 1e-30)
        HH  = -1.5*(1.0 + w_t)
        return [y, -(3.0+HH)*y - dV(rho,f,Lam4,alpha)/H2]
    return rhs

def run(f, Lam4, alpha, rho_ini, N_ini=-8.0, N_pts=6000):
    rhs  = make_rhs(f, Lam4, alpha)
    N_ev = np.linspace(N_ini, 0.0, N_pts)
    try:
        sol = solve_ivp(rhs, [N_ini, 0.0],
                        [rho_ini, 0.0],
                        t_eval=N_ev,
                        rtol=1e-11, atol=1e-14,
                        method='DOP853')
    except Exception:
        return None
    if sol.status != 0:
        return None

    a_arr   = np.exp(sol.t)
    rho_arr = sol.y[0]
    y_arr   = sol.y[1]
    w_arr   = np.zeros(N_pts)
    H2_arr  = np.zeros(N_pts)

    for i, N in enumerate(sol.t):
        rho = max(rho_arr[i], 1e-15)
        y   = y_arr[i]
        rr  = 3.0*Omega_r0*np.exp(-4*N)
        rm  = 3.0*Omega_m0*np.exp(-3*N)
        Vv  = V(rho, f, Lam4, alpha)
        den = max(1.0 - y**2/6.0, 1e-6)
        H2  = max((rr+rm+Vv)/(3.0*den), 1e-30)
        H2_arr[i] = H2
        KE  = 0.5*H2*y**2
        r_p = KE + Vv
        p_p = KE - Vv
        w_arr[i] = p_p/r_p if r_p > 1e-10 else -1.0

    # Omega_phi today
    rho0 = max(rho_arr[-1], 1e-15)
    Vv0  = V(rho0, f, Lam4, alpha)
    KE0  = 0.5*H2_arr[-1]*y_arr[-1]**2
    Op0  = (KE0+Vv0)/(3.0*H2_arr[-1])

    # Oscillation count
    rho_max_v = np.pi * f
    cross = int(np.sum(
        np.diff(np.sign(rho_arr - rho_max_v)) != 0))

    # w at specific scale factors
    a_check = [0.33, 0.50, 0.67, 0.80, 0.91, 1.00]
    w_at = {av: float(w_arr[np.argmin(np.abs(a_arr-av))])
            for av in a_check}

    # CPL fit
    mask = (a_arr >= 0.2) & (a_arr <= 1.0)
    if mask.sum() >= 4:
        A = np.column_stack([np.ones(mask.sum()),
                             1.0-a_arr[mask]])
        c,_,_,_ = np.linalg.lstsq(A,w_arr[mask],rcond=None)
        w0_cpl, wa_cpl = float(c[0]), float(c[1])
    else:
        w0_cpl, wa_cpl = None, None

    # True chi2: match actual w at observed redshifts
    # DESI measures distances at z = 0.51, 0.71, 0.93, 1.32
    # Corresponding a values:
    a_desi = [0.662, 0.585, 0.518, 0.431]
    w_desi_target = [w0_t + wa_t*(1.0-av)
                     for av in a_desi]
    # Also match w today
    w_chi2 = sum(
        (w_arr[np.argmin(np.abs(a_arr-av))] - wt)**2
        for av, wt in zip(a_desi, w_desi_target))
    w_chi2 += (w_at[1.00] - w0_t)**2  # w today = w0
    w_chi2 += (w_at[0.33] - (-1.0))**2 * 0.5  # early DE

    return dict(a=a_arr, rho=rho_arr, w=w_arr,
                w0=w0_cpl, wa=wa_cpl,
                Omega_phi0=Op0, crossings=cross,
                w_at=w_at, w_chi2=w_chi2)

# ═══════════════════════════════════════════════════════════
# STEP 1: Pure cosine — what is the best achievable?
# Establish the floor before adding alpha
# ═══════════════════════════════════════════════════════════
w0_t = -0.827
wa_t = -0.750

print()
print("STEP 1: Pure cosine (alpha=0) — best achievable?")
print("Optimizing for actual w(a) shape match")
print()

f_scan    = np.array([0.3,0.5,0.8,1.0,1.5,2.0,3.0,5.0])
Lam4_scan = np.logspace(-1, 2, 40)
rho_fracs = np.linspace(0.60, 0.99, 20)

pure_results = []
for fv in f_scan:
    rho_max = np.pi * fv
    for Lv in Lam4_scan:
        for rf in rho_fracs:
            r = run(fv, Lv, 0.0, rf*rho_max, N_pts=5000)
            if r is None:
                continue
            if r['crossings'] > 1:
                continue
            if abs(r['Omega_phi0'] - Omega_DE0) > 0.25:
                continue
            pure_results.append(
                (r['w_chi2'], fv, Lv, 0.0, rf,
                 r['w0'], r['wa'],
                 r['Omega_phi0'], r['w_at'],
                 r['a'].copy(), r['w'].copy()))

pure_results.sort(key=lambda x: x[0])
print(f"Pure cosine: {len(pure_results)} solutions")
if pure_results:
    b = pure_results[0]
    print(f"Best: w_today={b[8][1.00]:.4f}  "
          f"w₀_CPL={b[5]:.4f}  wa_CPL={b[6]:.4f}")
    print(f"f={b[1]}  Λ⁴={b[2]:.3e}  ρ_ini={b[4]:.3f}×πf")
    print(f"Shape chi2 = {b[0]:.5f}")

# ═══════════════════════════════════════════════════════════
# STEP 2: Two-term potential
# V = Lam4*(1-cos(rho/f) + alpha*(1-cos(2*rho/f)))
# alpha > 0 steepens the potential near the maximum
# alpha < 0 flattens it (creates a plateau = slower roll)
# ═══════════════════════════════════════════════════════════
print()
print("STEP 2: Two-term potential (alpha != 0)")
print("alpha < 0 creates plateau → slower late-time roll")
print()
print(f"{'f':>4}  {'Λ⁴':>7}  {'α':>6}  "
      f"{'ρ_ini/πf':>9}  "
      f"{'w(z=0)':>8}  {'w(z=0.5)':>9}  "
      f"{'w₀_CPL':>8}  {'wa_CPL':>8}  "
      f"{'Ω_φ':>6}  {'χ²_shape':>10}")
print("-"*90)

alpha_scan = [-0.30, -0.20, -0.15, -0.10,
              -0.05, -0.02, 0.0,
               0.02,  0.05,  0.10,  0.15, 0.20]

two_results = []
for fv in [0.3, 0.5, 0.8, 1.0, 1.5, 2.0]:
    rho_max = np.pi * fv
    for Lv in np.logspace(-1, 2, 30):
        for alpha in alpha_scan:
            for rf in np.linspace(0.60, 0.99, 15):
                r = run(fv, Lv, alpha,
                        rf*rho_max, N_pts=5000)
                if r is None:
                    continue
                if r['crossings'] > 1:
                    continue
                if abs(r['Omega_phi0']-Omega_DE0) > 0.25:
                    continue
                two_results.append(
                    (r['w_chi2'], fv, Lv, alpha, rf,
                     r['w0'], r['wa'],
                     r['Omega_phi0'], r['w_at'],
                     r['a'].copy(), r['w'].copy()))

two_results.sort(key=lambda x: x[0])
print(f"Two-term: {len(two_results)} solutions\n")

for row in two_results[:20]:
    chi2,fv,Lv,alpha,rf,w0,wa,Op,w_at,_,_ = row
    print(f"{fv:>4.1f}  {Lv:>7.2e}  {alpha:>6.3f}  "
          f"{rf:>9.3f}  "
          f"{w_at[1.00]:>8.4f}  {w_at[0.50]:>9.4f}  "
          f"{w0:>8.4f}  {wa:>8.4f}  "
          f"{Op:>6.4f}  {chi2:>10.5f}")

# ═══════════════════════════════════════════════════════════
# STEP 3: Shape comparison
# ═══════════════════════════════════════════════════════════
print()
print("STEP 3: Shape comparison at key redshifts")
print()
a_obs  = [0.33,0.50,0.67,0.80,0.91,1.00]
z_obs  = [2.03,1.00,0.49,0.25,0.10,0.00]
w_targ = [w0_t+wa_t*(1.0-av) for av in a_obs]

hdr = f"{'a':>5}  {'z':>4}  {'w_targ':>8}  "
hdr += f"{'pure_cos':>9}  "
for i in range(min(3, len(two_results))):
    hdr += f"{'2term_'+str(i+1):>10}  "
print(hdr)
print("-"*70)

for av, zv, wt in zip(a_obs, z_obs, w_targ):
    line = f"{av:>5.2f}  {zv:>4.2f}  {wt:>8.4f}  "
    # pure cosine best
    if pure_results:
        a_arr = pure_results[0][9]
        w_arr = pure_results[0][10]
        idx = np.argmin(np.abs(a_arr - av))
        line += f"{w_arr[idx]:>9.4f}  "
    for i in range(min(3, len(two_results))):
        a_arr = two_results[i][9]
        w_arr = two_results[i][10]
        idx = np.argmin(np.abs(a_arr - av))
        line += f"{w_arr[idx]:>10.4f}  "
    print(line)

# ═══════════════════════════════════════════════════════════
# PLOT
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
a_cpl = np.linspace(0.1, 1.0, 300)
w_cpl = w0_t + wa_t*(1.0 - a_cpl)
colors = ['red','blue','green','orange','purple']

# Top-left: w(a) comparison
ax = axes[0,0]
ax.plot(a_cpl, w_cpl, 'k-', lw=2.5,
        label='CPL target (w₀=-0.827, wa=-0.75)')
ax.axhline(-1.0, color='gray', ls=':', lw=1)
if pure_results:
    a_arr,w_arr = pure_results[0][9], pure_results[0][10]
    mask = a_arr >= 0.1
    ax.plot(a_arr[mask], w_arr[mask],
            'b--', lw=2, alpha=0.8,
            label=f'Pure cosine (best)')
for i, row in enumerate(two_results[:3]):
    a_arr,w_arr = row[9], row[10]
    mask = a_arr >= 0.1
    ax.plot(a_arr[mask], w_arr[mask],
            color=colors[i], lw=1.8, alpha=0.85,
            label=f'α={row[3]:.3f} f={row[1]} '
                  f'χ²={row[0]:.4f}')
ax.set_xlabel('a'); ax.set_ylabel('w(a)')
ax.set_title('w(a): Pure Cosine vs Two-Term Potential')
ax.set_xlim(0.1, 1.05); ax.set_ylim(-1.3, 0.1)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Top-right: w₀-wa plane
ax2 = axes[0,1]
theta = np.linspace(0, 2*np.pi, 200)
for ns, al in [(1,0.4),(2,0.2),(3,0.1)]:
    ax2.plot(w0_t+ns*0.09*np.cos(theta),
             wa_t+ns*0.33*np.sin(theta),
             'k-', lw=1.2, alpha=al)
ax2.plot(w0_t, wa_t, 'k*', ms=14, zorder=6,
         label='DESI target')
ax2.plot(-1.0, 0.0, 'g^', ms=8, label='ΛCDM')
if pure_results:
    ax2.scatter(pure_results[0][5], pure_results[0][6],
                s=120, marker='s', color='blue',
                zorder=5, label='Pure cosine best')
for i, row in enumerate(two_results[:20]):
    ax2.scatter(row[5], row[6],
                c=[row[0]], cmap='RdYlGn_r',
                vmin=0, vmax=0.2,
                s=50, zorder=3, alpha=0.7)
ax2.set_xlabel('w₀_CPL'); ax2.set_ylabel('wa_CPL')
ax2.set_title('w₀–wa: CPL Fit Values')
ax2.set_xlim(-1.2, -0.3); ax2.set_ylim(-1.2, 0.3)
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# Bottom-left: alpha effect on w_today
ax3 = axes[1,0]
for fv in [0.3, 0.5, 1.0]:
    alphas, w_today_list, chi2_list = [], [], []
    for alpha in alpha_scan:
        sub = [r for r in two_results
               if abs(r[1]-fv)<0.05
               and abs(r[3]-alpha)<0.005]
        if sub:
            best = min(sub, key=lambda x: x[0])
            alphas.append(alpha)
            w_today_list.append(best[8][1.00])
            chi2_list.append(best[0])
    if alphas:
        ax3.plot(alphas, w_today_list, 'o-',
                 label=f'f={fv}', lw=1.5)
ax3.axhline(w0_t, color='red', ls='--', lw=2,
            label=f'Target w₀={w0_t}')
ax3.axhline(-0.52, color='blue', ls=':', lw=1.5,
            label='Pure cosine today')
ax3.set_xlabel('α (two-term coefficient)')
ax3.set_ylabel('w(a=1) — actual value today')
ax3.set_title('Effect of α on w Today')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

# Bottom-right: potential shape
ax4 = axes[1,1]
rho_plot = np.linspace(0, np.pi, 200)
fv_plot, Lv_plot = 0.5, 1.0
for alpha, ls, label in [
        (0.0,   '-',  'α=0 (pure cosine)'),
        (-0.10, '--', 'α=-0.10'),
        (-0.20, ':',  'α=-0.20'),
        ( 0.10, '-.',  'α=+0.10')]:
    Vplot = V(rho_plot*fv_plot, fv_plot, Lv_plot, alpha)
    ax4.plot(rho_plot/np.pi, Vplot/Lv_plot,
             ls=ls, lw=2, label=label)
ax4.axvline(1.0, color='gray', ls=':', lw=1,
            label='ρ = πf (maximum)')
ax4.set_xlabel('ρ / πf')
ax4.set_ylabel('V / Λ⁴')
ax4.set_title('Potential Shape: Effect of α')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tafa_two_term.png', dpi=150,
            bbox_inches='tight')
print("\nSaved: tafa_two_term.png")

# ═══════════════════════════════════════════════════════════
# FINAL VERDICT
# ═══════════════════════════════════════════════════════════
print()
print("="*60)
print("FINAL VERDICT")
print("="*60)
print(f"\nTarget: w(a=1) = {w0_t}")
print()

if pure_results:
    pb = pure_results[0]
    print(f"Pure cosine best: w_today={pb[8][1.00]:.4f}"
          f"  shape_chi2={pb[0]:.5f}")

if two_results:
    tb = two_results[0]
    print(f"Two-term best:    w_today={tb[8][1.00]:.4f}"
          f"  shape_chi2={tb[0]:.5f}")
    print(f"  f={tb[1]}  Λ⁴={tb[2]:.3e}  "
          f"α={tb[3]:.3f}  ρ_ini={tb[4]:.3f}×πf")
    gap = abs(tb[8][1.00] - w0_t)
    print(f"\nRemaining gap in w_today: {gap:.4f}")
    if gap < 0.05:
        print("→ SOLVED. Two-term potential closes the gap.")
        print("  Alpha physically motivated: modifies axion")
        print("  potential via higher harmonics.")
    elif gap < 0.15:
        print("→ SIGNIFICANT IMPROVEMENT.")
        print("  Fine-tune α with denser grid.")
        print("  Physical interpretation: α = instanton")
        print("  correction to axion-like potential.")
    else:
        print("→ Two-term helps but gap remains.")
        print("  The cosine family cannot reproduce")
        print("  the required w(a) shape.")
        print("  HONEST CONCLUSION: V_cos and its")
        print("  harmonics produce a fixed w(a) shape.")
        print("  DESI requires a different shape.")
        print("  This model needs a different potential,")
        print("  or the TAFA dark energy connection")
        print("  requires rethinking from scratch.")

TAFA — HONEST TRAJECTORY MATCHING
Match actual w(a) shape, not just CPL fit

STEP 1: Pure cosine (alpha=0) — best achievable?
Optimizing for actual w(a) shape match

Pure cosine: 2867 solutions
Best: w_today=-0.8912  w₀_CPL=-0.9456  wa_CPL=-0.0835
f=0.3  Λ⁴=8.377e+00  ρ_ini=0.990×πf
Shape chi2 = 0.13523

STEP 2: Two-term potential (alpha != 0)
alpha < 0 creates plateau → slower late-time roll

   f       Λ⁴       α   ρ_ini/πf    w(z=0)   w(z=0.5)    w₀_CPL    wa_CPL     Ω_φ    χ²_shape
------------------------------------------------------------------------------------------
Two-term: 14080 solutions

 0.3  1.37e+00  -0.300      0.990   -0.8330    -0.9983   -0.9340   -0.1032  0.7403     0.12933
 0.3  2.21e+00  -0.200      0.990   -0.8162    -0.9979   -0.9237   -0.1191  0.8195     0.13035
 0.3  2.81e+00  -0.150      0.990   -0.8332    -0.9978   -0.9282   -0.1118  0.8526     0.13039
 0.3  3.56e+00  -0.100      0.990   -0.8658    -0.9979   -0.9397   -0.0936  0.8813     0.13161
 0.3  5.74e

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TAFA — CONE POPULATION EXTENSION")
print("Single-cone → packet-mediated Nc dynamics")
print("="*60)

# ── Cosmological parameters ──────────────────────────────
Omega_m0  = 0.300
Omega_r0  = 9.0e-5
Omega_DE0 = 1.0 - Omega_m0 - Omega_r0

# ── DESI target ──────────────────────────────────────────
w0_t = -0.827
wa_t = -0.750

# ════════════════════════════════════════════════════════
# PHASE 1: Single-cone baseline
# Extract packet energies → derive eps_star
# ════════════════════════════════════════════════════════

def V(rho, f, Lam4, alpha=0.0):
    x = rho / f
    return Lam4*(1.0 - np.cos(x)
                 + alpha*(1.0 - np.cos(2.0*x)))

def dV(rho, f, Lam4, alpha=0.0):
    x = rho / f
    return (Lam4/f)*(np.sin(x)
                     + 2.0*alpha*np.sin(2.0*x))

def V_twist(rho, a, n, beta=1.0):
    r = max(beta*rho, 1e-30)
    return n**2 / (2.0 * r**2 * a**6)

def single_cone_rhs(N, Y, f, Lam4, alpha, n):
    rho, y = Y
    a      = np.exp(N)
    rho    = max(rho, 1e-15)
    rr     = 3.0*Omega_r0*np.exp(-4*N)
    rm     = 3.0*Omega_m0*np.exp(-3*N)
    Vv     = V(rho, f, Lam4, alpha)
    Vt     = V_twist(rho, a, n)
    Vtot   = Vv + Vt
    den    = max(1.0 - y**2/6.0, 1e-6)
    H2     = max((rr + rm + Vtot)/(3.0*den), 1e-30)
    KE     = 0.5*H2*y**2
    r_phi  = KE + Vtot
    p_phi  = KE + Vt - Vv           # KE + Vtwist - Vpot
    w_tot  = (rr/3.0 + p_phi)/max(rr + rm + r_phi, 1e-30)
    HH     = -1.5*(1.0 + w_tot)
    # force from potential + twist centrifugal
    dVt_drho = -n**2 / (max(rho,1e-30)**3 * a**6)
    force  = dV(rho, f, Lam4, alpha) + dVt_drho
    drho   = y
    dy     = -(3.0 + HH)*y - force/H2
    return [drho, dy]

def run_single(f, Lam4, alpha, n, rho_ini,
               N_ini=-8.0, N_pts=8000):
    N_ev = np.linspace(N_ini, 0.0, N_pts)
    sol  = solve_ivp(
        lambda N,Y: single_cone_rhs(N,Y,f,Lam4,alpha,n),
        [N_ini, 0.0], [rho_ini, 0.0],
        t_eval=N_ev, rtol=1e-11, atol=1e-14,
        method='DOP853')
    if sol.status != 0:
        return None

    a_arr   = np.exp(sol.t)
    rho_arr = sol.y[0]
    y_arr   = sol.y[1]          # y = drho/dN
    N_pts_  = len(sol.t)

    w_arr  = np.zeros(N_pts_)
    H2_arr = np.zeros(N_pts_)
    Vt_arr = np.zeros(N_pts_)

    for i, N in enumerate(sol.t):
        a   = a_arr[i]
        rho = max(rho_arr[i], 1e-15)
        y   = y_arr[i]
        rr  = 3.0*Omega_r0*np.exp(-4*N)
        rm  = 3.0*Omega_m0*np.exp(-3*N)
        Vv  = V(rho, f, Lam4, alpha)
        Vt  = V_twist(rho, a, n)
        Vtot= Vv + Vt
        Vt_arr[i] = Vt
        den = max(1.0 - y**2/6.0, 1e-6)
        H2  = max((rr+rm+Vtot)/(3.0*den), 1e-30)
        H2_arr[i] = H2
        KE  = 0.5*H2*y**2
        r_p = KE + Vtot
        p_p = KE + Vt - Vv
        w_arr[i] = p_p/r_p if r_p > 1e-10 else -1.0

    # Omega_phi today
    rho0 = max(rho_arr[-1], 1e-15)
    Vv0  = V(rho0, f, Lam4, alpha)
    Vt0  = V_twist(rho0, 1.0, n)
    KE0  = 0.5*H2_arr[-1]*y_arr[-1]**2
    r_p0 = KE0 + Vv0 + Vt0
    Op0  = r_p0/(3.0*H2_arr[-1])

    return dict(N=sol.t, a=a_arr,
                rho=rho_arr, y=y_arr,
                w=w_arr, H2=H2_arr,
                Vt=Vt_arr, Op=Op0)

# ── Extract elongation packets from single-cone run ─────
def extract_packets(N_arr, a_arr, rho_arr, y_arr,
                    H2_arr, Vt_arr, f, Lam4, alpha, n,
                    eps_rel=0.02):
    """
    Elongation = rho actively decreasing fast enough.
    Q_geom  = max(-drho/dt, 0)  [geometric proxy]
    Q_twist = max(-dVt/dt, 0)   [twist-energy proxy]
    Packets = contiguous elongation episodes.
    """
    # drho/dt = y * H  (y = drho/dN, H = sqrt(H2))
    drho_dt  = y_arr * np.sqrt(H2_arr)
    dVt_dt   = np.gradient(Vt_arr, N_arr) * np.sqrt(H2_arr)

    # threshold: 2% of max elongation speed
    thresh = eps_rel * np.abs(drho_dt).max()

    in_elongation = drho_dt < -thresh

    packets_geom  = []
    packets_twist = []

    # integrate Q over contiguous episodes
    i = 0
    while i < len(N_arr):
        if in_elongation[i]:
            j = i
            while j < len(N_arr) and in_elongation[j]:
                j += 1
            # episode [i, j)
            dN = np.diff(N_arr[i:j+1])
            # Q_geom  = -drho/dN * sqrt(H2) (already drho_dt)
            Q_g = -drho_dt[i:j]
            Q_t = np.maximum(-dVt_dt[i:j], 0.0)
            a_ep= a_arr[i:j]
            E_g = np.trapz(Q_g,  N_arr[i:j])
            E_t = np.trapz(Q_t,  N_arr[i:j])
            packets_geom.append((
                a_arr[i], a_arr[j-1], E_g))
            packets_twist.append((
                a_arr[i], a_arr[j-1], E_t))
            i = j
        else:
            i += 1

    return packets_geom, packets_twist

# ════════════════════════════════════════════════════════
# PHASE 2: Extended model with Nc dynamics
# ════════════════════════════════════════════════════════

def extended_rhs(N, Y, f, Lam4, alpha, n,
                 eps_star, eps_thresh_frac=0.02):
    """
    Y = [rho, y, Nc]
    y = drho/dN
    """
    rho, y, Nc = Y
    a   = np.exp(N)
    rho = max(rho, 1e-15)
    Nc  = max(Nc,  1.0)

    rr   = 3.0*Omega_r0*np.exp(-4*N)
    rm   = 3.0*Omega_m0*np.exp(-3*N)
    Vv   = V(rho, f, Lam4, alpha)
    Vt   = V_twist(rho, a, n)
    Vtot = Vv + Vt

    den  = max(1.0 - y**2/6.0, 1e-6)
    # H2 uses total dark energy = Nc * single-cone
    rho_DE = Nc * Vtot / (1.0 - y**2/6.0)
    H2   = max((rr + rm + Nc*Vtot)/(3.0*den), 1e-30)
    H    = np.sqrt(H2)

    KE   = 0.5*H2*y**2
    r_p  = KE + Vtot
    p_p  = KE + Vt - Vv
    w_1  = p_p/r_p if r_p > 1e-10 else -1.0
    # Use Nc * r_p for Friedmann
    r_tot= rr + rm + Nc*r_p
    w_tot= (rr/3.0 + Nc*p_p)/max(r_tot, 1e-30)
    HH   = -1.5*(1.0 + w_tot)

    # Force on rho (same as single cone)
    dVt_drho = -n**2/(max(rho,1e-30)**3 * a**6)
    force    = dV(rho, f, Lam4, alpha) + dVt_drho

    drho = y
    dy   = -(3.0 + HH)*y - force/H2

    # Release proxy Q_geom = max(-drho/dt, 0)
    drho_dt = y * H
    thresh  = eps_thresh_frac  # absolute, tuned below
    Q = max(-drho_dt, 0.0) if drho_dt < -thresh else 0.0

    dNc = Q / eps_star if eps_star > 1e-30 else 0.0
    # dNc/dN = dNc/dt / H
    dNc_dN = dNc / H if H > 1e-30 else 0.0

    return [drho, dy, dNc_dN]

def run_extended(f, Lam4, alpha, n, rho_ini,
                 eps_star, eps_thresh=0.01,
                 N_ini=-8.0, N_pts=8000):
    N_ev = np.linspace(N_ini, 0.0, N_pts)

    def rhs(N, Y):
        return extended_rhs(N, Y, f, Lam4, alpha, n,
                            eps_star, eps_thresh)

    sol = solve_ivp(rhs, [N_ini, 0.0],
                    [rho_ini, 0.0, 1.0],
                    t_eval=N_ev,
                    rtol=1e-10, atol=1e-13,
                    method='DOP853')
    if sol.status != 0:
        return None

    a_arr  = np.exp(sol.t)
    rho_arr= sol.y[0]
    y_arr  = sol.y[1]
    Nc_arr = sol.y[2]
    N_pts_ = len(sol.t)

    w_arr  = np.zeros(N_pts_)
    H2_arr = np.zeros(N_pts_)

    for i, N_ in enumerate(sol.t):
        a   = a_arr[i]
        rho = max(rho_arr[i], 1e-15)
        y   = y_arr[i]
        Nc  = max(Nc_arr[i], 1.0)
        rr  = 3.0*Omega_r0*np.exp(-4*N_)
        rm  = 3.0*Omega_m0*np.exp(-3*N_)
        Vv  = V(rho, f, Lam4, alpha)
        Vt  = V_twist(rho, a, n)
        Vtot= Vv + Vt
        den = max(1.0 - y**2/6.0, 1e-6)
        H2  = max((rr+rm+Nc*Vtot)/(3.0*den), 1e-30)
        H2_arr[i] = H2
        KE  = 0.5*H2*y**2
        r_p = KE + Vtot
        p_p = KE + Vt - Vv
        # effective w: use Nc-weighted dark energy
        r_DE= Nc*r_p
        p_DE= Nc*p_p
        w_arr[i] = p_DE/r_DE if r_DE > 1e-10 else -1.0

    rho0 = max(rho_arr[-1], 1e-15)
    Vv0  = V(rho0, f, Lam4, alpha)
    Vt0  = V_twist(rho0, 1.0, n)
    KE0  = 0.5*H2_arr[-1]*y_arr[-1]**2
    Nc0  = max(Nc_arr[-1], 1.0)
    r_p0 = KE0 + Vv0 + Vt0
    Op0  = Nc0*r_p0/(3.0*H2_arr[-1])

    # CPL fit
    mask = (a_arr >= 0.2) & (a_arr <= 1.0)
    w0_c, wa_c = None, None
    if mask.sum() >= 4:
        A = np.column_stack([np.ones(mask.sum()),
                             1.0-a_arr[mask]])
        c,_,_,_ = np.linalg.lstsq(
            A, w_arr[mask], rcond=None)
        w0_c, wa_c = float(c[0]), float(c[1])

    return dict(a=a_arr, rho=rho_arr,
                w=w_arr, Nc=Nc_arr,
                H2=H2_arr, Op=Op0,
                w0=w0_c, wa=wa_c,
                w_today=w_arr[-1])

# ════════════════════════════════════════════════════════
# RUN SEQUENCE
# ════════════════════════════════════════════════════════

# Best parameters from previous scan
BEST = [
    dict(f=0.3, Lam4=1.374, alpha=-0.30,
         n=0, rho_frac=0.990),
    dict(f=0.3, Lam4=2.21,  alpha=-0.20,
         n=0, rho_frac=0.990),
    dict(f=0.3, Lam4=2.81,  alpha=-0.15,
         n=0, rho_frac=0.990),
    # n=1 variants for twist dynamics
    dict(f=0.5, Lam4=3.16,  alpha=0.0,
         n=1, rho_frac=0.950),
    dict(f=0.3, Lam4=2.15,  alpha=0.0,
         n=1, rho_frac=0.980),
]

print()
print("─"*60)
print("PHASE 1: Single-cone baselines + packet extraction")
print("─"*60)
print()

baselines = []
for cfg in BEST:
    f, L, al, n = (cfg['f'], cfg['Lam4'],
                   cfg['alpha'], cfg['n'])
    rho_ini = cfg['rho_frac'] * np.pi * f
    r = run_single(f, L, al, n, rho_ini)
    if r is None:
        print(f"f={f} n={n}: integration failed")
        continue
    if abs(r['Op'] - Omega_DE0) > 0.30:
        print(f"f={f} n={n}: Ω_φ={r['Op']:.3f} out of range")
        continue

    pg, pt = extract_packets(
        r['N'], r['a'], r['rho'], r['y'],
        r['H2'], r['Vt'], f, L, al, n)

    E_g = np.array([p[2] for p in pg]) if pg else np.array([])
    E_t = np.array([p[2] for p in pt]) if pt else np.array([])

    # CPL fit
    mask = (r['a'] >= 0.2) & (r['a'] <= 1.0)
    w0_c, wa_c = None, None
    if mask.sum() >= 4:
        A = np.column_stack([np.ones(mask.sum()),
                             1.0-r['a'][mask]])
        c,_,_,_ = np.linalg.lstsq(
            A, r['w'][mask], rcond=None)
        w0_c, wa_c = float(c[0]), float(c[1])

    print(f"f={f:>4.1f}  Λ⁴={L:.3f}  "
          f"α={al:>6.3f}  n={n}")
    print(f"  Ω_φ={r['Op']:.4f}  "
          f"w_today={r['w'][-1]:.4f}  "
          f"w₀={w0_c:.4f}  wa={wa_c:.4f}")
    print(f"  Geom packets: {len(E_g)}  "
          + (f"E_g: min={E_g.min():.4f} "
             f"med={np.median(E_g):.4f} "
             f"max={E_g.max():.4f}"
             if len(E_g) > 0 else "none"))
    print(f"  Twist packets: {len(E_t)}  "
          + (f"E_t: min={E_t.min():.4f} "
             f"med={np.median(E_t):.4f} "
             f"max={E_t.max():.4f}"
             if len(E_t) > 0 else "none"))
    print()

    eps_g = float(np.median(E_g)) if len(E_g) > 0 else None
    eps_t = float(np.median(E_t)) if len(E_t) > 0 else None

    baselines.append(dict(cfg=cfg, result=r,
                          pkts_g=pg, pkts_t=pt,
                          E_g=E_g, E_t=E_t,
                          eps_g=eps_g, eps_t=eps_t,
                          w0=w0_c, wa=wa_c))

# ════════════════════════════════════════════════════════
# PHASE 2: Extended Nc model
# ════════════════════════════════════════════════════════
print()
print("─"*60)
print("PHASE 2: Extended model with Nc population dynamics")
print("─"*60)
print()
print(f"{'f':>4}  {'Λ⁴':>6}  {'α':>6}  {'n':>2}  "
      f"{'ε★ src':>8}  {'ε★':>8}  "
      f"{'Nc(0)':>6}  "
      f"{'w_today':>8}  {'w₀':>8}  {'wa':>8}  "
      f"{'Ω_φ':>6}  {'dist':>6}")
print("─"*90)

ext_results = []

for b in baselines:
    cfg = b['cfg']
    f, L, al, n = (cfg['f'], cfg['Lam4'],
                   cfg['alpha'], cfg['n'])
    rho_ini = cfg['rho_frac'] * np.pi * f

    for src, eps in [('geom',  b['eps_g']),
                     ('twist', b['eps_t'])]:
        if eps is None or eps < 1e-10:
            continue

        # Also try eps * 0.5 and eps * 2.0
        for eps_scale in [0.25, 0.5, 1.0, 2.0, 4.0]:
            eps_use = eps * eps_scale
            r = run_extended(f, L, al, n, rho_ini,
                             eps_use, eps_thresh=0.005)
            if r is None:
                continue
            if r['w0'] is None:
                continue
            if abs(r['Op'] - Omega_DE0) > 0.35:
                continue

            chi2 = ((r['w0']-w0_t)**2
                    + (r['wa']-wa_t)**2)
            dist = np.sqrt(chi2)

            Nc_today = float(r['Nc'][-1])
            tag = f"{src}×{eps_scale}"

            print(f"{f:>4.1f}  {L:>6.3f}  {al:>6.3f}  "
                  f"{n:>2}  {tag:>8}  "
                  f"{eps_use:>8.4f}  "
                  f"{Nc_today:>6.2f}  "
                  f"{r['w_today']:>8.4f}  "
                  f"{r['w0']:>8.4f}  {r['wa']:>8.4f}  "
                  f"{r['Op']:>6.4f}  {dist:>6.4f}")

            ext_results.append(dict(
                f=f, L=L, al=al, n=n,
                src=src, eps_scale=eps_scale,
                eps=eps_use, dist=dist,
                result=r, baseline=b))

# ════════════════════════════════════════════════════════
# PHASE 3: Shape table and diagnostics
# ════════════════════════════════════════════════════════
ext_results.sort(key=lambda x: x['dist'])

print()
print("─"*60)
print("PHASE 3: Shape table — best extended vs baselines")
print("─"*60)
print()

a_check = [0.33,0.50,0.67,0.80,0.91,1.00]
z_check = [2.03,1.00,0.49,0.25,0.10,0.00]
w_targ  = [w0_t+wa_t*(1.0-av) for av in a_check]

hdr = f"{'a':>5}  {'z':>4}  {'target':>8}  "
if baselines:
    hdr += f"{'1cone':>8}  "
for i in range(min(3, len(ext_results))):
    hdr += f"{'ext_'+str(i+1):>9}  "
print(hdr)
print("─"*70)

for av, zv, wt in zip(a_check, z_check, w_targ):
    line = f"{av:>5.2f}  {zv:>4.2f}  {wt:>8.4f}  "
    if baselines:
        a0 = baselines[0]['result']['a']
        w0 = baselines[0]['result']['w']
        line += f"{w0[np.argmin(np.abs(a0-av))]:>8.4f}  "
    for i in range(min(3, len(ext_results))):
        ae = ext_results[i]['result']['a']
        we = ext_results[i]['result']['w']
        line += f"{we[np.argmin(np.abs(ae-av))]:>9.4f}  "
    print(line)

# ════════════════════════════════════════════════════════
# PLOTS
# ════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
a_cpl = np.linspace(0.1, 1.0, 300)
w_cpl = w0_t + wa_t*(1.0 - a_cpl)
colors = ['red','blue','green','orange','purple']

# ── w(a) ────────────────────────────────────────────────
ax = axes[0,0]
ax.plot(a_cpl, w_cpl, 'k-', lw=2.5,
        label=f'DESI CPL target')
ax.axhline(-1.0, color='gray', ls=':', lw=1)
if baselines:
    b0 = baselines[0]
    mask = b0['result']['a'] >= 0.1
    ax.plot(b0['result']['a'][mask],
            b0['result']['w'][mask],
            'b--', lw=2, alpha=0.8,
            label=f"1-cone: w₀={b0['w0']:.3f}")
for i, er in enumerate(ext_results[:3]):
    r = er['result']
    mask = r['a'] >= 0.1
    ax.plot(r['a'][mask], r['w'][mask],
            color=colors[i], lw=1.8, alpha=0.85,
            label=f"Nc: ε={er['eps']:.3f} "
                  f"w₀={r['w0']:.3f} "
                  f"wa={r['wa']:.3f}")
ax.set_xlabel('a'); ax.set_ylabel('w(a)')
ax.set_title('w(a): Single-cone vs Nc Extended')
ax.set_xlim(0.1,1.05); ax.set_ylim(-1.3,0.2)
ax.legend(fontsize=7); ax.grid(True,alpha=0.3)

# ── Nc(a) ────────────────────────────────────────────────
ax2 = axes[0,1]
for i, er in enumerate(ext_results[:4]):
    r = er['result']
    mask = r['a'] >= 0.1
    ax2.plot(r['a'][mask], r['Nc'][mask],
             color=colors[i], lw=1.8,
             label=f"ε={er['eps']:.3f} "
                   f"({er['src']})")
ax2.axhline(1.0, color='gray', ls=':', lw=1,
            label='Nc=1 baseline')
ax2.set_xlabel('a'); ax2.set_ylabel('Nc(a)')
ax2.set_title('Cone Population Growth')
ax2.set_xlim(0.1,1.05)
ax2.legend(fontsize=8); ax2.grid(True,alpha=0.3)

# ── w0-wa plane ──────────────────────────────────────────
ax3 = axes[0,2]
theta = np.linspace(0,2*np.pi,200)
for ns, al_ in [(1,0.40),(2,0.20),(3,0.10)]:
    ax3.plot(w0_t+ns*0.09*np.cos(theta),
             wa_t+ns*0.33*np.sin(theta),
             'k-', lw=1.2, alpha=al_)
ax3.plot(w0_t, wa_t, 'k*', ms=14, zorder=6,
         label='DESI target')
ax3.plot(-1.0, 0.0, 'g^', ms=8, label='ΛCDM')
if baselines:
    b0 = baselines[0]
    ax3.scatter(b0['w0'], b0['wa'],
                s=140, marker='s', color='blue',
                zorder=5, label='1-cone best')
for i, er in enumerate(ext_results[:20]):
    r = er['result']
    ax3.scatter(r['w0'], r['wa'],
                c=[er['dist']], cmap='RdYlGn_r',
                vmin=0, vmax=0.8,
                s=60, zorder=3, alpha=0.8)
ax3.set_xlabel('w₀'); ax3.set_ylabel('wa')
ax3.set_title('w₀–wa: Nc Extension')
ax3.set_xlim(-1.2,-0.3); ax3.set_ylim(-1.2,0.5)
ax3.legend(fontsize=8); ax3.grid(True,alpha=0.3)

# ── Packet energies ──────────────────────────────────────
ax4 = axes[1,0]
for i, b in enumerate(baselines[:3]):
    Eg = b['E_g']
    if len(Eg) > 0:
        ax4.bar(np.arange(len(Eg)) + i*0.25,
                Eg, width=0.22, alpha=0.7,
                color=colors[i],
                label=f"f={b['cfg']['f']} geom")
ax4.axhline(0, color='gray', ls=':', lw=1)
ax4.set_xlabel('Packet index k')
ax4.set_ylabel('ΔE_k (geometric proxy)')
ax4.set_title('Release Packet Energies')
ax4.legend(fontsize=8); ax4.grid(True,alpha=0.3)

# ── dist vs eps_scale ────────────────────────────────────
ax5 = axes[1,1]
for b in baselines[:3]:
    cfg = b['cfg']
    fv  = cfg['f']
    for src in ['geom','twist']:
        pts = [(er['eps_scale'], er['dist'])
               for er in ext_results
               if er['f']==fv and er['src']==src]
        if pts:
            pts.sort(key=lambda x: x[0])
            xs = [p[0] for p in pts]
            ys = [p[1] for p in pts]
            lbl = f"f={fv} {src}"
            ax5.plot(xs, ys, 'o-', lw=1.5,
                     label=lbl)
ax5.axhline(0.09, color='red', ls='--', lw=1.5,
            label='~1σ threshold')
ax5.set_xlabel('ε★ scale factor')
ax5.set_ylabel('Distance to DESI target')
ax5.set_title('Distance vs ε★ Scaling')
ax5.legend(fontsize=8); ax5.grid(True,alpha=0.3)
ax5.set_xscale('log')

# ── ρ(a) trajectory ──────────────────────────────────────
ax6 = axes[1,2]
if baselines:
    b0 = baselines[0]
    r0 = b0['result']
    mask = r0['a'] >= 0.1
    ax6.plot(r0['a'][mask],
             r0['rho'][mask]/(np.pi*b0['cfg']['f']),
             'b--', lw=2, label='1-cone ρ/πf')
if ext_results:
    r1 = ext_results[0]['result']
    mask = r1['a'] >= 0.1
    ax6.plot(r1['a'][mask],
             r1['rho'][mask]/(np.pi*ext_results[0]['f']),
             'r-', lw=2, label='Nc ext ρ/πf')
ax6.axhline(1.0, color='gray', ls=':', lw=1,
            label='ρ = πf (potential max)')
ax6.set_xlabel('a'); ax6.set_ylabel('ρ / πf')
ax6.set_title('Field Trajectory')
ax6.set_xlim(0.1,1.05)
ax6.legend(fontsize=8); ax6.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('tafa_Nc_extension.png', dpi=150,
            bbox_inches='tight')
print()
print("Saved: tafa_Nc_extension.png")

# ════════════════════════════════════════════════════════
# ════════════════════════════════════════════════════════
# VERDICT
# ════════════════════════════════════════════════════════
print()
print("="*60)
print("VERDICT")
print("="*60)

if baselines:
    b0 = baselines[0]
    d0 = np.sqrt((b0['w0']-w0_t)**2
                 + (b0['wa']-wa_t)**2)
    print(f"\n1-cone best:   w₀={b0['w0']:.4f}  "
          f"wa={b0['wa']:.4f}  dist={d0:.4f}")

if ext_results:
    er = ext_results[0]
    r  = er['result']
    print(f"Nc-ext best:   w₀={r['w0']:.4f}  "
          f"wa={r['wa']:.4f}  dist={er['dist']:.4f}")
    print(f"  f={er['f']}  Λ⁴={er['L']:.3f}  "
          f"α={er['al']:.3f}  n={er['n']}")
    print(f"  ε★={er['eps']:.4f}  "
          f"source={er['src']}×{er['eps_scale']}")
    print(f"  Nc today = {r['Nc'][-1]:.3f}")
    print()
    imp = d0 - er['dist'] if baselines else 0
    if er['dist'] < 0.10:
        print("→ WITHIN 1σ. Nc dynamics closes the gap.")
        print("  Check Nc growth is physically reasonable.")
        print("  Check packet energies cluster (integer test).")
    elif er['dist'] < 0.20:
        print(f"→ Improvement of {imp:.4f} over 1-cone.")
        print("  Nc helps. Refine ε★ with finer grid.")
    else:
        print("→ Nc extension does not improve significantly.")
        print("  The w0-wa target may be outside the")
        print("  reachable region of this potential class.")

print()
print("Packet clustering test:")
for b in baselines[:2]:
    Eg = b['E_g']
    if len(Eg) > 1:
        cv = Eg.std()/Eg.mean() if Eg.mean()>0 else 999
        print(f"  f={b['cfg']['f']} geom: "
              f"{len(Eg)} packets  "
              f"CV={cv:.3f}  "
              f"({'clustered ✓' if cv<0.5 else 'spread ✗'})")

TAFA — CONE POPULATION EXTENSION
Single-cone → packet-mediated Nc dynamics

────────────────────────────────────────────────────────────
PHASE 1: Single-cone baselines + packet extraction
────────────────────────────────────────────────────────────

f= 0.3  Λ⁴=1.374  α=-0.300  n=0
  Ω_φ=0.7403  w_today=-0.8330  w₀=-0.9341  wa=-0.1031
  Geom packets: 1  E_g: min=0.1956 med=0.1956 max=0.1956
  Twist packets: 1  E_t: min=0.0000 med=0.0000 max=0.0000

f= 0.3  Λ⁴=2.210  α=-0.200  n=0
  Ω_φ=0.8194  w_today=-0.8166  w₀=-0.9240  wa=-0.1186
  Geom packets: 1  E_g: min=0.2695 med=0.2695 max=0.2695
  Twist packets: 1  E_t: min=0.0000 med=0.0000 max=0.0000

f= 0.3  Λ⁴=2.810  α=-0.150  n=0
  Ω_φ=0.8527  w_today=-0.8328  w₀=-0.9282  wa=-0.1118
  Geom packets: 1  E_g: min=0.3015 med=0.3015 max=0.3015
  Twist packets: 1  E_t: min=0.0000 med=0.0000 max=0.0000

f= 0.5  Λ⁴=3.160  α= 0.000  n=1
  Ω_φ=0.4864  w_today=0.9423  w₀=0.4359  wa=0.5727
  Geom packets: 0  none
  Twist packets: 0  none

f= 0.3  Λ⁴=

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TAFA — PHASE 4: DISCRETE PACKET-THRESHOLD DYNAMICS")
print("Backreaction-enabled burst model")
print("="*60)

# ── Cosmological constants ───────────────────────────────
Omega_m0  = 0.300
Omega_r0  = 9.0e-5
Omega_DE0 = 1.0 - Omega_m0 - Omega_r0
w0_t, wa_t = -0.827, -0.750

# ════════════════════════════════════════════════════════
# CORE POTENTIALS
# ════════════════════════════════════════════════════════
def V(rho, f, Lam4, alpha=0.0):
    x = rho / f
    return Lam4*(1.0 - np.cos(x)
                 + alpha*(1.0 - np.cos(2.0*x)))

def dV(rho, f, Lam4, alpha=0.0):
    x = rho / f
    return (Lam4/f)*(np.sin(x)
                     + 2.0*alpha*np.sin(2.0*x))

def V_twist(rho, a, n, beta=1.0):
    r = max(abs(beta*rho), 1e-30)
    return n**2 / (2.0 * r**2 * max(a,1e-30)**6)

# ════════════════════════════════════════════════════════
# PHASE 4 INTEGRATOR
# Uses continuous ODE + discrete event layer
# State: [rho, y, E_avail, A_accum]
#   rho     : radial field
#   y       : drho/dN
#   E_avail : remaining twist energy budget
#   A_accum : accumulator toward next packet
# Discrete events update packet count externally
# ════════════════════════════════════════════════════════

class PacketEngine:
    """
    Manages discrete packet events during integration.
    Accumulator A grows when Q > 0 (elongation).
    When A >= eps_star: fire a packet event.
    Backreaction: reduce E_avail by eps_star,
                  damp y by gamma (momentum reduction).
    """
    def __init__(self, eps_star, gamma=0.85,
                 eps_thresh=0.005):
        self.eps_star   = eps_star
        self.gamma      = gamma          # velocity damping
        self.eps_thresh = eps_thresh     # elongation threshold
        self.N_pkt      = 0              # packet count
        self.A          = 0.0            # accumulator
        self.events     = []             # (N, E_avail, Npkt)
        self.fired      = False          # flag this step

    def reset(self):
        self.N_pkt  = 0
        self.A      = 0.0
        self.events = []
        self.fired  = False

    def step(self, N, rho, y, H, E_avail):
        """
        Called every ODE step.
        Returns (dA_dN, backreact_dy, backreact_dE).
        All backreaction terms are zero unless event fires.
        """
        # Q_geom: release proxy (drho/dt < 0)
        drho_dt = y * H
        if drho_dt < -self.eps_thresh:
            Q = -drho_dt              # positive quantity
        else:
            Q = 0.0

        # accumulator grows as dA/dN = Q/H
        dA_dN = Q / max(H, 1e-30)

        self.fired = False
        d_y      = 0.0
        d_E      = 0.0

        # Check threshold (can fire multiple times)
        while (self.A >= self.eps_star
               and self.eps_star > 1e-30
               and E_avail > self.eps_star * 0.1):
            self.A      -= self.eps_star
            E_avail     -= self.eps_star
            self.N_pkt  += 1
            d_E         -= self.eps_star
            # velocity damping at packet event
            d_y          = y * (self.gamma - 1.0)
            self.fired   = True
            self.events.append((N, E_avail, self.N_pkt))

        return dA_dN, d_y, d_E, E_avail

def make_packet_rhs(f, Lam4, alpha, n, engine):
    """
    Returns RHS for [rho, y, E_avail, A_accum].
    Discrete events handled via engine side-effects.
    """
    def rhs(N, Y):
        rho, y, E_av, A = Y
        a   = np.exp(N)
        rho = max(rho, 1e-15)
        E_av= max(E_av, 0.0)
        A   = max(A,    0.0)

        rr   = 3.0*Omega_r0*np.exp(-4*N)
        rm   = 3.0*Omega_m0*np.exp(-3*N)
        Vv   = V(rho,  f, Lam4, alpha)
        # Twist energy: use remaining budget E_av
        # as effective amplitude rescaling
        Vt_raw = V_twist(rho, a, n)
        E_init = engine._E_init if hasattr(
            engine,'_E_init') else max(Vt_raw,1e-30)
        scale  = E_av / max(E_init, 1e-30)
        Vt     = Vt_raw * max(scale, 0.0)
        Vtot   = Vv + Vt

        den  = max(1.0 - y**2/6.0, 1e-6)
        H2   = max((rr+rm+Vtot)/(3.0*den), 1e-30)
        H    = np.sqrt(H2)
        KE   = 0.5*H2*y**2
        r_p  = KE + Vtot
        p_p  = KE + Vt - Vv
        w_t  = (rr/3.0 + p_p)/max(rr+rm+r_p, 1e-30)
        HH   = -1.5*(1.0 + w_t)

        dVt_drho = (-n**2 /
                    (max(rho,1e-30)**3 * a**6)) * max(scale,0.0)
        force  = dV(rho,f,Lam4,alpha) + dVt_drho
        drho   = y
        dy     = -(3.0+HH)*y - force/H2

        # Packet engine step
        dA_dN, d_y, d_E, E_av_new = engine.step(
            N, rho, y, H, E_av)

        # Apply backreaction instantly (impulse)
        dy   += d_y      # velocity damping
        dE_dN = (E_av_new - E_av)  # instantaneous
        dA    = dA_dN

        return [drho, dy, dE_dN, dA]
    return rhs

def run_packet_model(f, Lam4, alpha, n,
                     rho_ini, eps_star, gamma=0.85,
                     eps_thresh=0.005,
                     N_ini=-8.0, N_pts=10000):
    engine = PacketEngine(eps_star, gamma, eps_thresh)

    # Compute initial twist energy for budget
    a_ini   = np.exp(N_ini)
    Vt_init = V_twist(rho_ini, a_ini, n)
    engine._E_init = max(Vt_init, 1e-30)
    engine.reset()

    N_ev = np.linspace(N_ini, 0.0, N_pts)
    rhs  = make_packet_rhs(f, Lam4, alpha, n, engine)

    Y0 = [rho_ini, 0.0, Vt_init, 0.0]
    try:
        sol = solve_ivp(rhs, [N_ini, 0.0], Y0,
                        t_eval=N_ev,
                        rtol=1e-9, atol=1e-12,
                        method='DOP853',
                        max_step=0.02)
    except Exception as e:
        return None

    if sol.status != 0 and sol.status != 1:
        return None

    a_arr  = np.exp(sol.t)
    rho_arr= sol.y[0]
    y_arr  = sol.y[1]
    Eav_arr= sol.y[2]
    A_arr  = sol.y[3]
    Npts_  = len(sol.t)

    w_arr  = np.zeros(Npts_)
    H2_arr = np.zeros(Npts_)

    for i, N_ in enumerate(sol.t):
        a   = a_arr[i]
        rho = max(rho_arr[i], 1e-15)
        y   = y_arr[i]
        E_av= max(Eav_arr[i], 0.0)
        rr  = 3.0*Omega_r0*np.exp(-4*N_)
        rm  = 3.0*Omega_m0*np.exp(-3*N_)
        Vv  = V(rho,f,Lam4,alpha)
        Vt_r= V_twist(rho,a,n)
        sc  = E_av/max(engine._E_init,1e-30)
        Vt  = Vt_r*max(sc,0.0)
        Vtot= Vv+Vt
        den = max(1.0-y**2/6.0,1e-6)
        H2  = max((rr+rm+Vtot)/(3.0*den),1e-30)
        H2_arr[i]=H2
        KE  = 0.5*H2*y**2
        r_p = KE+Vtot
        p_p = KE+Vt-Vv
        w_arr[i] = p_p/r_p if r_p>1e-10 else -1.0

    # Omega_phi
    rho0 = max(rho_arr[-1],1e-15)
    Vv0  = V(rho0,f,Lam4,alpha)
    Vt0  = V_twist(rho0,1.0,n)
    sc0  = max(Eav_arr[-1],0.0)/max(engine._E_init,1e-30)
    KE0  = 0.5*H2_arr[-1]*y_arr[-1]**2
    Op0  = (KE0+Vv0+Vt0*sc0)/(3.0*H2_arr[-1])

    # CPL fit
    mask = (a_arr>=0.2)&(a_arr<=1.0)
    w0_c=wa_c=None
    if mask.sum()>=4:
        A = np.column_stack([np.ones(mask.sum()),
                             1.0-a_arr[mask]])
        c,_,_,_ = np.linalg.lstsq(
            A,w_arr[mask],rcond=None)
        w0_c,wa_c = float(c[0]),float(c[1])

    return dict(a=a_arr, rho=rho_arr, y=y_arr,
                w=w_arr, H2=H2_arr,
                E_avail=Eav_arr, A_acc=A_arr,
                w0=w0_c, wa=wa_c,
                w_today=w_arr[-1],
                Op=Op0,
                N_pkt=engine.N_pkt,
                events=engine.events,
                engine=engine)

# ════════════════════════════════════════════════════════
# REFERENCE: clean single-cone (no packets)
# ════════════════════════════════════════════════════════
def run_single_clean(f, Lam4, alpha, n, rho_ini,
                     N_ini=-8.0, N_pts=8000):
    def rhs(N, Y):
        rho,y = Y
        a   = np.exp(N)
        rho = max(rho,1e-15)
        rr  = 3.0*Omega_r0*np.exp(-4*N)
        rm  = 3.0*Omega_m0*np.exp(-3*N)
        Vv  = V(rho,f,Lam4,alpha)
        Vt  = V_twist(rho,a,n)
        Vtot= Vv+Vt
        den = max(1.0-y**2/6.0,1e-6)
        H2  = max((rr+rm+Vtot)/(3.0*den),1e-30)
        KE  = 0.5*H2*y**2
        r_p = KE+Vtot
        p_p = KE+Vt-Vv
        w_t = (rr/3.0+p_p)/max(rr+rm+r_p,1e-30)
        HH  = -1.5*(1.0+w_t)
        dVt_dr = -n**2/(max(rho,1e-30)**3*a**6)
        force  = dV(rho,f,Lam4,alpha)+dVt_dr
        return [y, -(3.0+HH)*y - force/H2]

    N_ev = np.linspace(N_ini,0.0,N_pts)
    sol  = solve_ivp(rhs,[N_ini,0.0],[rho_ini,0.0],
                     t_eval=N_ev,rtol=1e-11,atol=1e-14,
                     method='DOP853')
    if sol.status!=0: return None
    a_arr  = np.exp(sol.t)
    rho_arr= sol.y[0]
    y_arr  = sol.y[1]
    w_arr  = np.zeros(N_pts)
    H2_arr = np.zeros(N_pts)
    for i,N_ in enumerate(sol.t):
        a,rho,y = a_arr[i],max(rho_arr[i],1e-15),y_arr[i]
        rr=3.0*Omega_r0*np.exp(-4*N_)
        rm=3.0*Omega_m0*np.exp(-3*N_)
        Vv=V(rho,f,Lam4,alpha)
        Vt=V_twist(rho,a,n)
        Vtot=Vv+Vt
        den=max(1.0-y**2/6.0,1e-6)
        H2=max((rr+rm+Vtot)/(3.0*den),1e-30)
        H2_arr[i]=H2
        KE=0.5*H2*y**2
        r_p=KE+Vtot; p_p=KE+Vt-Vv
        w_arr[i]=p_p/r_p if r_p>1e-10 else -1.0
    mask=(a_arr>=0.2)&(a_arr<=1.0)
    w0_c=wa_c=None
    if mask.sum()>=4:
        A=np.column_stack([np.ones(mask.sum()),
                           1.0-a_arr[mask]])
        c,_,_,_=np.linalg.lstsq(A,w_arr[mask],rcond=None)
        w0_c,wa_c=float(c[0]),float(c[1])
    Op0_=(0.5*H2_arr[-1]*y_arr[-1]**2
          +V(max(rho_arr[-1],1e-15),f,Lam4,alpha)
          +V_twist(max(rho_arr[-1],1e-15),1.0,n)
          )/(3.0*H2_arr[-1])
    return dict(a=a_arr,w=w_arr,
                w0=w0_c,wa=wa_c,
                w_today=w_arr[-1],Op=Op0_)

# ════════════════════════════════════════════════════════
# PARAMETER GRID
# ════════════════════════════════════════════════════════
CONFIGS = [
    # best two-term from Phase 2
    dict(f=0.3,  Lam4=1.374, alpha=-0.30,
         n=0, rho_frac=0.990),
    dict(f=0.3,  Lam4=2.210, alpha=-0.20,
         n=0, rho_frac=0.990),
    dict(f=0.3,  Lam4=2.810, alpha=-0.15,
         n=0, rho_frac=0.990),
    # n=1 variants: twist energy present → real packets
    dict(f=0.5,  Lam4=3.160, alpha=0.00,
         n=1, rho_frac=0.950),
    dict(f=0.3,  Lam4=2.150, alpha=0.00,
         n=1, rho_frac=0.980),
    dict(f=0.3,  Lam4=1.374, alpha=-0.30,
         n=1, rho_frac=0.990),
    # wider f scan with n=1
    dict(f=0.5,  Lam4=1.500, alpha=-0.20,
         n=1, rho_frac=0.970),
    dict(f=0.8,  Lam4=5.000, alpha=-0.10,
         n=1, rho_frac=0.960),
]

eps_star_fracs = [0.05, 0.10, 0.20, 0.40, 0.80]
gamma_vals     = [0.60, 0.75, 0.85, 0.95]

# ════════════════════════════════════════════════════════
# STEP 1: Baselines
# ════════════════════════════════════════════════════════
print()
print("─"*60)
print("STEP 1: Single-cone references")
print("─"*60)
print(f"{'f':>4} {'Λ⁴':>7} {'α':>6} {'n':>2} "
      f"{'w_today':>8} {'w₀':>8} {'wa':>8} {'Ω_φ':>6}")
print("─"*55)

baselines = {}
for cfg in CONFIGS:
    f,L,al,n = (cfg['f'],cfg['Lam4'],
                cfg['alpha'],cfg['n'])
    ri = cfg['rho_frac']*np.pi*f
    r  = run_single_clean(f,L,al,n,ri)
    key= (f,L,al,n,cfg['rho_frac'])
    if r is None:
        print(f"{f:>4.1f} {L:>7.3f} {al:>6.3f} "
              f"{n:>2} FAILED")
        continue
    baselines[key] = r
    print(f"{f:>4.1f} {L:>7.3f} {al:>6.3f} {n:>2} "
          f"{r['w_today']:>8.4f} "
          f"{r['w0']:>8.4f} {r['wa']:>8.4f} "
          f"{r['Op']:>6.4f}")

# ════════════════════════════════════════════════════════
# STEP 2: Packet scan
# ════════════════════════════════════════════════════════
print()
print("─"*60)
print("STEP 2: Discrete packet-threshold scan")
print("─"*60)
print(f"{'f':>4} {'Λ⁴':>7} {'α':>6} {'n':>2} "
      f"{'ε★':>7} {'γ':>5} "
      f"{'Npkt':>5} "
      f"{'w_today':>8} {'w₀':>8} {'wa':>8} "
      f"{'Ω_φ':>6} {'dist':>6}")
print("─"*85)

all_results = []
for cfg in CONFIGS:
    f,L,al,n = (cfg['f'],cfg['Lam4'],
                cfg['alpha'],cfg['n'])
    ri  = cfg['rho_frac']*np.pi*f
    key = (f,L,al,n,cfg['rho_frac'])
    if key not in baselines:
        continue
    b   = baselines[key]

    # Estimate eps_star from initial twist energy
    a_ini  = np.exp(-8.0)
    Vt_ini = V_twist(ri, a_ini, n)
    E_scale= max(Vt_ini, 1e-6)

    for ef in eps_star_fracs:
        eps = ef * E_scale
        for gam in gamma_vals:
            r = run_packet_model(
                f,L,al,n,ri,eps,gam,
                eps_thresh=0.003)
            if r is None or r['w0'] is None:
                continue
            if abs(r['Op']-Omega_DE0)>0.35:
                continue
            dist = np.sqrt((r['w0']-w0_t)**2
                           +(r['wa']-wa_t)**2)
            print(f"{f:>4.1f} {L:>7.3f} {al:>6.3f} "
                  f"{n:>2} "
                  f"{eps:>7.4f} {gam:>5.2f} "
                  f"{r['N_pkt']:>5} "
                  f"{r['w_today']:>8.4f} "
                  f"{r['w0']:>8.4f} {r['wa']:>8.4f} "
                  f"{r['Op']:>6.4f} {dist:>6.4f}")
            all_results.append(dict(
                cfg=cfg, eps=eps, gam=gam,
                result=r, dist=dist,
                baseline=b))

# ════════════════════════════════════════════════════════
# STEP 3: Shape table
# ════════════════════════════════════════════════════════
all_results.sort(key=lambda x: x['dist'])

print()
print("─"*60)
print("STEP 3: Shape comparison at DESI redshifts")
print("─"*60)

a_obs = [0.33,0.50,0.67,0.80,0.91,1.00]
z_obs = [2.03,1.00,0.49,0.25,0.10,0.00]
w_tg  = [w0_t+wa_t*(1.0-av) for av in a_obs]

hdr = f"{'a':>5} {'z':>4} {'target':>8}"
# pick one baseline
b0key = list(baselines.keys())[0]
hdr += f" {'1cone':>8}"
for i in range(min(3,len(all_results))):
    hdr += f" {'pkt'+str(i+1):>9}"
print(hdr); print("─"*65)

for av,zv,wt in zip(a_obs,z_obs,w_tg):
    line = f"{av:>5.2f} {zv:>4.2f} {wt:>8.4f}"
    b0r = baselines[b0key]
    line += f" {b0r['w'][np.argmin(np.abs(b0r['a']-av))]:>8.4f}"
    for i in range(min(3,len(all_results))):
        ae = all_results[i]['result']['a']
        we = all_results[i]['result']['w']
        line += f" {we[np.argmin(np.abs(ae-av))]:>9.4f}"
    print(line)

# ════════════════════════════════════════════════════════
# PLOTS
# ════════════════════════════════════════════════════════
fig,axes = plt.subplots(2,3,figsize=(18,11))
a_cpl = np.linspace(0.1,1.0,300)
w_cpl = w0_t + wa_t*(1.0-a_cpl)
C = ['red','blue','green','orange','purple']

# ── w(a) comparison ──────────────────────────────────────
ax=axes[0,0]
ax.plot(a_cpl,w_cpl,'k-',lw=2.5,
        label='DESI CPL target')
ax.axhline(-1.0,color='gray',ls=':',lw=1)
b0r=baselines[b0key]
mask=b0r['a']>=0.1
ax.plot(b0r['a'][mask],b0r['w'][mask],
        'b--',lw=2,alpha=0.8,
        label=f"1-cone  w₀={b0r['w0']:.3f} "
              f"wa={b0r['wa']:.3f}")
for i,er in enumerate(all_results[:4]):
    r=er['result']
    mask=r['a']>=0.1
    ax.plot(r['a'][mask],r['w'][mask],
            color=C[i],lw=1.8,alpha=0.85,
            label=(f"Npkt={r['N_pkt']} "
                   f"γ={er['gam']:.2f} "
                   f"w₀={r['w0']:.3f} "
                   f"wa={r['wa']:.3f}"))
ax.set_xlabel('a'); ax.set_ylabel('w(a)')
ax.set_title('w(a): 1-cone vs Packet Model')
ax.set_xlim(0.1,1.05); ax.set_ylim(-1.3,0.2)
ax.legend(fontsize=7); ax.grid(True,alpha=0.3)

# ── w₀-wa plane ──────────────────────────────────────────
ax2=axes[0,1]
theta=np.linspace(0,2*np.pi,200)
for ns,al_ in [(1,0.4),(2,0.2),(3,0.1)]:
    ax2.plot(w0_t+ns*0.09*np.cos(theta),
             wa_t+ns*0.33*np.sin(theta),
             'k-',lw=1.2,alpha=al_)
ax2.plot(w0_t,wa_t,'k*',ms=14,zorder=6,
         label='DESI target')
ax2.plot(-1.0,0.0,'g^',ms=8,label='ΛCDM')
for bkey,bval in list(baselines.items())[:4]:
    ax2.scatter(bval['w0'],bval['wa'],
                s=120,marker='s',color='blue',
                zorder=5,alpha=0.6)
sc=ax2.scatter(
    [er['result']['w0'] for er in all_results[:60]],
    [er['result']['wa'] for er in all_results[:60]],
    c=[er['dist'] for er in all_results[:60]],
    cmap='RdYlGn_r',vmin=0,vmax=0.8,
    s=50,zorder=3,alpha=0.8)
plt.colorbar(sc,ax=ax2,label='dist to target')
ax2.set_xlabel('w₀'); ax2.set_ylabel('wa')
ax2.set_title('w₀–wa: Packet Model Coverage')
ax2.set_xlim(-1.2,-0.3); ax2.set_ylim(-1.2,0.5)
ax2.legend(fontsize=7); ax2.grid(True,alpha=0.3)

# ── E_avail(a) decay ─────────────────────────────────────
ax3=axes[0,2]
for i,er in enumerate(all_results[:4]):
    r=er['result']
    mask=r['a']>=0.1
    ax3.plot(r['a'][mask],r['E_avail'][mask],
             color=C[i],lw=1.8,
             label=(f"Npkt={r['N_pkt']} "
                    f"ε★={er['eps']:.4f}"))
ax3.set_xlabel('a')
ax3.set_ylabel('E_avail(a)')
ax3.set_title('Available Twist Budget Decay')
ax3.set_xlim(0.1,1.05)
ax3.legend(fontsize=8); ax3.grid(True,alpha=0.3)

# ── wa vs N_pkt ──────────────────────────────────────────
ax4=axes[1,0]
npkts = [er['result']['N_pkt'] for er in all_results]
was   = [er['result']['wa']    for er in all_results]
dists = [er['dist']            for er in all_results]
sc2=ax4.scatter(npkts,was,c=dists,
                cmap='RdYlGn_r',vmin=0,vmax=0.8,
                s=50,alpha=0.8)
ax4.axhline(wa_t,color='red',ls='--',lw=2,
            label=f'DESI wa={wa_t}')
ax4.axhline(baselines[b0key]['wa'],
            color='blue',ls=':',lw=1.5,
            label='1-cone wa')
plt.colorbar(sc2,ax=ax4,label='dist')
ax4.set_xlabel('N_pkt (discrete events)')
ax4.set_ylabel('wa_CPL')
ax4.set_title('wa vs Packet Count')
ax4.legend(fontsize=8); ax4.grid(True,alpha=0.3)

# ── dist vs gamma ────────────────────────────────────────
ax5=axes[1,1]
for ef in eps_star_fracs[:4]:
    pts=[]
    for er in all_results:
        if abs(er['eps']/max(er['eps'],1e-30)
               -ef)<0.01 or True:
            pass
    # group by gamma
    for gam in sorted(set(er['gam']
                          for er in all_results)):
        sub=[er for er in all_results
             if abs(er['gam']-gam)<0.01]
        if sub:
            best=min(sub,key=lambda x:x['dist'])
            pts.append((gam,best['dist']))
    if pts:
        pts.sort()
        ax5.plot([p[0] for p in pts],
                 [p[1] for p in pts],'o-',
                 lw=1.5,label=f'best/γ')
        break
# all gamma vs dist
for i,(cfg,grp) in enumerate(
    {str(er['cfg']):[] for er in all_results}.items()):
    pass
gam_arr = sorted(set(er['gam'] for er in all_results))
dist_med= [np.median([er['dist'] for er in all_results
                       if er['gam']==g])
           for g in gam_arr]
ax5.plot(gam_arr,dist_med,'ro-',lw=2,
         label='median dist')
ax5.axhline(0.09,color='gray',ls='--',
            label='~1σ')
ax5.set_xlabel('γ (velocity damping)')
ax5.set_ylabel('Distance to DESI target')
ax5.set_title('Backreaction Strength vs Distance')
ax5.legend(fontsize=8); ax5.grid(True,alpha=0.3)

# ── Accumulator A(a) ─────────────────────────────────────
ax6=axes[1,2]
for i,er in enumerate(all_results[:4]):
    r=er['result']
    mask=r['a']>=0.1
    ax6.plot(r['a'][mask],r['A_acc'][mask],
             color=C[i],lw=1.8,
             label=f"ε★={er['eps']:.4f}")
    # mark event times
    for ev in er['result']['events']:
        N_ev_,_,_ = ev
        a_ev = np.exp(N_ev_)
        ax6.axvline(a_ev,color=C[i],
                    ls=':',lw=0.8,alpha=0.6)
ax6.set_xlabel('a')
ax6.set_ylabel('Accumulator A(a)')
ax6.set_title('Packet Accumulator (dashed = event)')
ax6.set_xlim(0.1,1.05)
ax6.legend(fontsize=8); ax6.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('tafa_phase4_packets.png',dpi=150,
            bbox_inches='tight')
print("\nSaved: tafa_phase4_packets.png")

# ════════════════════════════════════════════════════════
# VERDICT
# ════════════════════════════════════════════════════════
print()
print("="*60)
print("VERDICT")
print("="*60)
b0r=baselines[b0key]
d0=np.sqrt((b0r['w0']-w0_t)**2+(b0r['wa']-wa_t)**2)
print(f"\n1-cone reference: w₀={b0r['w0']:.4f} "
      f"wa={b0r['wa']:.4f}  dist={d0:.4f}")
if all_results:
    best=all_results[0]
    r=best['result']
    print(f"Packet best:      w₀={r['w0']:.4f} "
          f"wa={r['wa']:.4f}  dist={best['dist']:.4f}")
    print(f"  f={best['cfg']['f']}  "
          f"Λ⁴={best['cfg']['Lam4']:.3f}  "
          f"α={best['cfg']['alpha']:.3f}  "
          f"n={best['cfg']['n']}")
    print(f"  ε★={best['eps']:.5f}  "
          f"γ={best['gam']:.2f}  "
          f"Npkt={r['N_pkt']}")
    imp = d0 - best['dist']
    print(f"\nImprovement over 1-cone: Δdist={imp:.4f}")
    if best['dist'] < 0.10:
        print("→ WITHIN ~1σ DESI contour.")
        print("  Verify: are Npkt events physically reasonable?")
        print("  Verify: does wa gain come from packet timing")
        print("  or from Ω_φ shift?")
    elif imp > 0.05:
        print("→ Meaningful improvement. Refine ε★ and γ.")
        print("  Focus on n=1 configs: twist energy")
        print("  gives real packet structure.")
    else:
        print("→ Packets alone insufficient.")
        print("  The w(a) shape constraint is binding.")
        print("  wa requires faster late-time evolution")
        print("  than this potential class supports.")
        print("\nHonest physical conclusion:")
        print("  Standard cosine quintessence, even with")
        print("  packet backreaction, cannot produce")
        print("  wa ~ -0.75 without either:")
        print("  (a) crossing w = -1 [phantom],")
        print("  (b) a sharply different potential shape,")
        print("  or (c) the DESI CPL fit being a poor")
        print("  description of the actual data.")

print()
print("Packet event log (best solution):")
if all_results:
    for ev in all_results[0]['result']['events'][:10]:
        N_e,E_e,Np_e = ev
        print(f"  a={np.exp(N_e):.4f}  "
              f"E_avail={E_e:.5f}  "
              f"Npkt={Np_e}")
    if len(all_results[0]['result']['events'])==0:
        print("  No packet events fired.")
        print("  ε★ may be too large or Q too small.")
        print("  Try n>=1 to activate twist energy."),

TAFA — PHASE 4: DISCRETE PACKET-THRESHOLD DYNAMICS
Backreaction-enabled burst model

────────────────────────────────────────────────────────────
STEP 1: Single-cone references
────────────────────────────────────────────────────────────
   f      Λ⁴      α  n  w_today       w₀       wa    Ω_φ
───────────────────────────────────────────────────────
 0.3   1.374 -0.300  0  -0.8330  -0.9341  -0.1031 0.7403
 0.3   2.210 -0.200  0  -0.8166  -0.9240  -0.1186 0.8194
 0.3   2.810 -0.150  0  -0.8328  -0.9282  -0.1118 0.8527
 0.5   3.160  0.000  1   0.9423   0.4359   0.5727 0.4864
 0.3   2.150  0.000  1  -0.4590   0.4656   0.7365 0.5893
 0.3   1.374 -0.300  1   0.6681   0.9176   0.0771 0.4406
 0.5   1.500 -0.200  1   0.9929   0.8347   0.1297 0.2795
 0.8   5.000 -0.100  1  -0.9219  -1.5928   3.2225 0.9126

────────────────────────────────────────────────────────────
STEP 2: Discrete packet-threshold scan
────────────────────────────────────────────────────────────
   f      Λ⁴      α  n      ε★ 

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TAFA — PHASE 5: GEOMETRIC TWIST GATING")
print("Non-constant β(ρ) with n=1 controlled release")
print("="*60)

# ── Cosmological constants ───────────────────────────────
Omega_m0  = 0.300
Omega_r0  = 9.0e-5
Omega_DE0 = 1.0 - Omega_m0 - Omega_r0
w0_t, wa_t = -0.827, -0.750

# ════════════════════════════════════════════════════════
# β(ρ) FAMILY — three physically motivated forms
# ════════════════════════════════════════════════════════
def beta_const(rho, params):
    """Baseline: β = 1 everywhere"""
    return 1.0, 0.0          # (β, dβ/dρ)

def beta_powerlaw(rho, params):
    """
    β(ρ) = (ρ/(ρ+ρ_c))^s
    Small at small ρ → twist suppressed early
    Saturates near 1 at large ρ
    """
    rho_c = params['rho_c']
    s     = params['s']
    r     = max(rho, 1e-30)
    u     = r / (r + rho_c)
    b     = u**s
    dbdr  = s * u**(s-1) * rho_c / (r+rho_c)**2
    return b, dbdr

def beta_sigmoid(rho, params):
    """
    β(ρ) = β_min + (1-β_min)·σ(k(ρ-ρ_c))
    Smooth step: suppressed for ρ < ρ_c
    """
    rho_c   = params['rho_c']
    k       = params['k']
    b_min   = params.get('b_min', 0.05)
    r       = max(rho, 1e-30)
    sig     = 1.0/(1.0+np.exp(-k*(r-rho_c)))
    b       = b_min + (1.0-b_min)*sig
    dbdr    = (1.0-b_min)*k*sig*(1.0-sig)
    return b, dbdr

def beta_throat(rho, params):
    """
    β(ρ) = β_min + (1-β_min)·exp(-(ρ-ρ_c)²/(2σ²))
    Peak suppression at ρ_c — 'throat' geometry
    Twist energy amplified near ρ_c
    """
    rho_c = params['rho_c']
    sig   = params['sigma']
    b_min = params.get('b_min', 0.1)
    r     = max(rho, 1e-30)
    g     = np.exp(-0.5*((r-rho_c)/max(sig,1e-10))**2)
    b     = b_min + (1.0-b_min)*g
    dbdr  = -(1.0-b_min)*g*(r-rho_c)/max(sig**2,1e-20)
    return b, dbdr

BETA_FORMS = {
    'const':     beta_const,
    'powerlaw':  beta_powerlaw,
    'sigmoid':   beta_sigmoid,
    'throat':    beta_throat,
}

# ════════════════════════════════════════════════════════
# POTENTIALS
# ════════════════════════════════════════════════════════
def V(rho, f, Lam4, alpha=0.0):
    x = rho/f
    return Lam4*(1.0 - np.cos(x)
                 + alpha*(1.0 - np.cos(2.0*x)))

def dV(rho, f, Lam4, alpha=0.0):
    x = rho/f
    return (Lam4/f)*(np.sin(x)
                     + 2.0*alpha*np.sin(2.0*x))

def V_twist_beta(rho, a, n, beta_fn, beta_params):
    b,_ = beta_fn(rho, beta_params)
    b   = max(b, 1e-6)
    r   = max(rho, 1e-30)
    return n**2 / (2.0*b**2*r**2*max(a,1e-30)**6)

def dVt_drho(rho, a, n, beta_fn, beta_params):
    """∂V_twist/∂ρ  including β'(ρ) contribution"""
    b,db = beta_fn(rho, beta_params)
    b    = max(b, 1e-6)
    r    = max(rho, 1e-30)
    a6   = max(a, 1e-30)**6
    # V_t = n²/(2 b² ρ² a⁶)
    # dVt/dρ = -n²/(b² ρ² a⁶) · (1/ρ + db/b/dρ · ... wait
    # = n²/(2 a⁶) · d/dρ [1/(b(ρ)²ρ²)]
    # = n²/(2 a⁶) · [-2/(b³ρ²) db/dρ - 2/(b²ρ³)]
    pref = n**2 / (2.0*a6)
    term1 = -2.0*db / (b**3 * r**2)
    term2 = -2.0       / (b**2 * r**3)
    return pref*(term1 + term2)

# ════════════════════════════════════════════════════════
# ODE WITH β(ρ) AND PACKET ENGINE
# State: [ρ, y, E_av, A]
# ════════════════════════════════════════════════════════
class GatedPacketEngine:
    def __init__(self, eps_star, gamma,
                 rho_lo, rho_hi, eps_thresh=0.003):
        self.eps_star   = eps_star
        self.gamma      = gamma
        self.rho_lo     = rho_lo
        self.rho_hi     = rho_hi
        self.eps_thresh = eps_thresh
        self.N_pkt      = 0
        self.A          = 0.0
        self.events     = []

    def reset(self):
        self.N_pkt=0; self.A=0.0; self.events=[]

    def step(self, N, rho, y, H, E_av):
        drho_dt = y*H
        in_band = (self.rho_lo <= rho <= self.rho_hi)
        if drho_dt < -self.eps_thresh and in_band:
            Q = -drho_dt
        else:
            Q = 0.0
        dA_dN = Q/max(H,1e-30)

        d_y=0.0; d_E=0.0
        while (self.A >= self.eps_star
               and self.eps_star > 1e-30
               and E_av > self.eps_star*0.05):
            self.A  -= self.eps_star
            E_av    -= self.eps_star
            self.N_pkt += 1
            d_E     -= self.eps_star
            d_y      = y*(self.gamma-1.0)
            self.events.append((N, E_av, self.N_pkt))
        return dA_dN, d_y, d_E, E_av

def make_gated_rhs(f, Lam4, alpha, n,
                   beta_fn, beta_params, engine):
    def rhs(N, Y):
        rho,y,E_av,A = Y
        a    = np.exp(N)
        rho  = max(rho,  1e-15)
        E_av = max(E_av, 0.0)
        A    = max(A,    0.0)

        rr  = 3.0*Omega_r0*np.exp(-4*N)
        rm  = 3.0*Omega_m0*np.exp(-3*N)
        Vv  = V(rho, f, Lam4, alpha)

        # Twist: scale by remaining budget
        Vt_raw = V_twist_beta(rho,a,n,beta_fn,beta_params)
        E0     = engine._E_init
        scale  = E_av/max(E0,1e-30) if E0>0 else 0.0
        scale  = max(min(scale,1.0),0.0)
        Vt     = Vt_raw * scale
        Vtot   = Vv + Vt

        den  = max(1.0-y**2/6.0,1e-6)
        H2   = max((rr+rm+Vtot)/(3.0*den),1e-30)
        H    = np.sqrt(H2)
        KE   = 0.5*H2*y**2
        r_p  = KE+Vtot; p_p = KE+Vt-Vv
        w_t  = (rr/3.0+p_p)/max(rr+rm+r_p,1e-30)
        HH   = -1.5*(1.0+w_t)

        force_phi = dV(rho,f,Lam4,alpha)
        force_t   = dVt_drho(rho,a,n,
                              beta_fn,beta_params)*scale
        dy = -(3.0+HH)*y - (force_phi+force_t)/H2

        dA_dN,d_y,d_E,E_av_new = engine.step(
            N,rho,y,H,E_av)

        return [y, dy+d_y, E_av_new-E_av, dA_dN]
    return rhs

def run_gated(f, Lam4, alpha, n,
              rho_ini, beta_fn, beta_params,
              eps_frac, gamma,
              rho_lo_frac=0.3, rho_hi_frac=0.95,
              N_ini=-8.0, N_pts=10000):

    a_ini  = np.exp(N_ini)
    Vt_ini = V_twist_beta(rho_ini,a_ini,n,
                          beta_fn,beta_params)
    E_init = max(Vt_ini, 1e-8)
    eps_s  = eps_frac * E_init
    rho_max= np.pi*f
    rlo    = rho_lo_frac*rho_max
    rhi    = rho_hi_frac*rho_max

    eng = GatedPacketEngine(eps_s,gamma,rlo,rhi)
    eng._E_init = E_init
    eng.reset()

    rhs  = make_gated_rhs(f,Lam4,alpha,n,
                          beta_fn,beta_params,eng)
    N_ev = np.linspace(N_ini,0.0,N_pts)
    Y0   = [rho_ini, 0.0, E_init, 0.0]

    try:
        sol = solve_ivp(rhs,[N_ini,0.0],Y0,
                        t_eval=N_ev,
                        rtol=1e-9,atol=1e-12,
                        method='DOP853',
                        max_step=0.02)
    except Exception:
        return None
    if sol.status not in (0,1):
        return None

    a_arr  = np.exp(sol.t)
    rho_arr= sol.y[0]; y_arr = sol.y[1]
    Eav_arr= sol.y[2]; A_arr = sol.y[3]
    Np_    = len(sol.t)

    w_arr=np.zeros(Np_); H2_arr=np.zeros(Np_)
    for i,N_ in enumerate(sol.t):
        a   = a_arr[i]; rho = max(rho_arr[i],1e-15)
        y   = y_arr[i];  Eav = max(Eav_arr[i],0.0)
        rr  = 3.0*Omega_r0*np.exp(-4*N_)
        rm  = 3.0*Omega_m0*np.exp(-3*N_)
        Vv  = V(rho,f,Lam4,alpha)
        Vt_r= V_twist_beta(rho,a,n,beta_fn,beta_params)
        sc  = max(min(Eav/max(E_init,1e-30),1.0),0.0)
        Vt  = Vt_r*sc; Vtot=Vv+Vt
        den = max(1.0-y**2/6.0,1e-6)
        H2  = max((rr+rm+Vtot)/(3.0*den),1e-30)
        H2_arr[i]=H2
        KE=0.5*H2*y**2; r_p=KE+Vtot; p_p=KE+Vt-Vv
        w_arr[i]=p_p/r_p if r_p>1e-10 else -1.0

    rho0=max(rho_arr[-1],1e-15)
    Vv0 =V(rho0,f,Lam4,alpha)
    sc0 =max(min(Eav_arr[-1]/max(E_init,1e-30),1.),0.)
    Vt0 =V_twist_beta(rho0,1.,n,beta_fn,beta_params)*sc0
    KE0 =0.5*H2_arr[-1]*y_arr[-1]**2
    Op0 =(KE0+Vv0+Vt0)/(3.0*H2_arr[-1])

    mask=(a_arr>=0.2)&(a_arr<=1.0)
    w0_c=wa_c=None
    if mask.sum()>=4:
        A_=np.column_stack([np.ones(mask.sum()),
                            1.0-a_arr[mask]])
        c,_,_,_=np.linalg.lstsq(A_,w_arr[mask],
                                  rcond=None)
        w0_c,wa_c=float(c[0]),float(c[1])

    return dict(a=a_arr,rho=rho_arr,w=w_arr,
                H2=H2_arr,Eav=Eav_arr,A=A_arr,
                w0=w0_c,wa=wa_c,
                w_today=w_arr[-1],Op=Op0,
                N_pkt=eng.N_pkt,
                events=eng.events,
                eps_star=eps_s,
                E_init=E_init)

# ════════════════════════════════════════════════════════
# SCAN DESIGN
# ════════════════════════════════════════════════════════
BASE_CONFIGS = [
    dict(f=0.3, Lam4=1.374, alpha=-0.30,
         n=1, rho_frac=0.990),
    dict(f=0.3, Lam4=2.210, alpha=-0.20,
         n=1, rho_frac=0.990),
    dict(f=0.5, Lam4=3.160, alpha= 0.00,
         n=1, rho_frac=0.950),
    dict(f=0.5, Lam4=1.500, alpha=-0.20,
         n=1, rho_frac=0.970),
    dict(f=0.8, Lam4=5.000, alpha=-0.10,
         n=1, rho_frac=0.960),
]

BETA_CONFIGS = [
    ('const',    {}),
    ('powerlaw', {'rho_c': 0.5, 's': 1.0}),
    ('powerlaw', {'rho_c': 0.3, 's': 2.0}),
    ('powerlaw', {'rho_c': 0.8, 's': 0.5}),
    ('sigmoid',  {'rho_c': 0.5, 'k': 8.0,  'b_min':0.05}),
    ('sigmoid',  {'rho_c': 0.7, 'k': 5.0,  'b_min':0.10}),
    ('sigmoid',  {'rho_c': 0.3, 'k':12.0,  'b_min':0.02}),
    ('throat',   {'rho_c': 0.6, 'sigma':0.2,'b_min':0.10}),
    ('throat',   {'rho_c': 0.4, 'sigma':0.3,'b_min':0.05}),
]

eps_fracs  = [0.05, 0.10, 0.20, 0.40]
gammas     = [0.70, 0.80, 0.90]
rlo_fracs  = [0.20, 0.40, 0.60]

# ════════════════════════════════════════════════════════
# RUN SCAN
# ════════════════════════════════════════════════════════
print()
print("─"*70)
print("Scan: β(ρ) form × (f,Λ,α) × ε★ × γ × band")
print("─"*70)
print(f"{'β-form':>10} {'f':>4} {'n':>2} "
      f"{'ε_f':>5} {'γ':>5} "
      f"{'Npkt':>5} "
      f"{'w_today':>8} {'w₀':>8} {'wa':>8} "
      f"{'Ω_φ':>6} {'dist':>6}")
print("─"*75)

all_results = []
for cfg in BASE_CONFIGS:
    f,L,al,n,rf = (cfg['f'],cfg['Lam4'],
                   cfg['alpha'],cfg['n'],
                   cfg['rho_frac'])
    rho_ini = rf*np.pi*f
    for bname, bparams in BETA_CONFIGS:
        # scale rho_c by actual πf if present
        bp = dict(bparams)
        if 'rho_c' in bp:
            bp['rho_c'] *= np.pi*f
        if 'sigma' in bp:
            bp['sigma'] *= np.pi*f
        beta_fn = BETA_FORMS[bname]
        for ef in eps_fracs:
            for gam in gammas:
                for rlof in rlo_fracs:
                    r = run_gated(
                        f,L,al,n,rho_ini,
                        beta_fn,bp,ef,gam,
                        rho_lo_frac=rlof,
                        rho_hi_frac=0.97)
                    if r is None or r['w0'] is None:
                        continue
                    if abs(r['Op']-Omega_DE0)>0.35:
                        continue
                    if r['w_today'] > -0.50:
                        continue   # unphysical
                    dist=np.sqrt((r['w0']-w0_t)**2
                                 +(r['wa']-wa_t)**2)
                    tag = f"{bname[:6]}"
                    print(
                        f"{tag:>10} {f:>4.1f} {n:>2} "
                        f"{ef:>5.2f} {gam:>5.2f} "
                        f"{r['N_pkt']:>5} "
                        f"{r['w_today']:>8.4f} "
                        f"{r['w0']:>8.4f} "
                        f"{r['wa']:>8.4f} "
                        f"{r['Op']:>6.4f} "
                        f"{dist:>6.4f}")
                    all_results.append(dict(
                        bname=bname, bp=bp,
                        cfg=cfg, ef=ef, gam=gam,
                        rlof=rlof, dist=dist,
                        result=r))

all_results.sort(key=lambda x: x['dist'])

# ════════════════════════════════════════════════════════
# SHAPE TABLE
# ════════════════════════════════════════════════════════
print()
print("─"*60)
print("Shape comparison at DESI redshifts")
print("─"*60)
a_obs=[0.33,0.50,0.67,0.80,0.91,1.00]
z_obs=[2.03,1.00,0.49,0.25,0.10,0.00]
w_tg =[w0_t+wa_t*(1-av) for av in a_obs]
hdr  =f"{'a':>5} {'z':>4} {'target':>8}"
for i in range(min(4,len(all_results))):
    hdr+=f" {'best'+str(i+1):>9}"
print(hdr); print("─"*65)
for av,zv,wt in zip(a_obs,z_obs,w_tg):
    line=f"{av:>5.2f} {zv:>4.2f} {wt:>8.4f}"
    for i in range(min(4,len(all_results))):
        ae=all_results[i]['result']['a']
        we=all_results[i]['result']['w']
        line+=f" {we[np.argmin(np.abs(ae-av))]:>9.4f}"
    print(line)

# ════════════════════════════════════════════════════════
# β(ρ) DIAGNOSTIC: plot the β profiles
# ════════════════════════════════════════════════════════
fig,axes=plt.subplots(2,3,figsize=(18,11))
C=['red','blue','green','orange','purple','brown']

# Panel 1: β(ρ) profiles
ax=axes[0,0]
rho_plot=np.linspace(0.01,np.pi*0.5,300)
for bname,bparams in BETA_CONFIGS:
    bp=dict(bparams)
    if 'rho_c' in bp: bp['rho_c']*=np.pi*0.3
    if 'sigma' in bp: bp['sigma']*=np.pi*0.3
    bfn=BETA_FORMS[bname]
    bvals=[bfn(r,bp)[0] for r in rho_plot]
    ax.plot(rho_plot/( np.pi*0.3),bvals,
            lw=1.5,alpha=0.8,
            label=f"{bname}")
ax.axhline(1.0,color='gray',ls=':')
ax.set_xlabel('ρ/(πf)'); ax.set_ylabel('β(ρ)')
ax.set_title('β(ρ) Profiles (f=0.3)')
ax.legend(fontsize=7); ax.grid(True,alpha=0.3)

# Panel 2: w(a) best solutions
ax2=axes[0,1]
a_cpl=np.linspace(0.1,1.0,300)
w_cpl=w0_t+wa_t*(1-a_cpl)
ax2.plot(a_cpl,w_cpl,'k-',lw=2.5,label='DESI CPL')
ax2.axhline(-1,color='gray',ls=':')
for i,er in enumerate(all_results[:5]):
    r=er['result']
    mask=r['a']>=0.1
    ax2.plot(r['a'][mask],r['w'][mask],
             color=C[i%6],lw=1.8,alpha=0.85,
             label=(f"{er['bname'][:5]} "
                    f"Npkt={r['N_pkt']} "
                    f"w₀={r['w0']:.3f} "
                    f"wa={r['wa']:.3f}"))
ax2.set_xlabel('a'); ax2.set_ylabel('w(a)')
ax2.set_title('w(a): Best Gated Solutions')
ax2.set_xlim(0.1,1.05); ax2.set_ylim(-1.3,0.2)
ax2.legend(fontsize=7); ax2.grid(True,alpha=0.3)

# Panel 3: w₀-wa plane
ax3=axes[0,2]
theta=np.linspace(0,2*np.pi,200)
for ns,al_ in [(1,0.4),(2,0.2),(3,0.1)]:
    ax3.plot(w0_t+ns*0.09*np.cos(theta),
             wa_t+ns*0.33*np.sin(theta),
             'k-',lw=1.2,alpha=al_)
ax3.plot(w0_t,wa_t,'k*',ms=14,zorder=6,
         label='DESI target')
ax3.plot(-1.0,0.0,'g^',ms=8,label='ΛCDM')
# color by β form
bnames=list({er['bname'] for er in all_results})
cmap_b={b:C[i%6] for i,b in enumerate(bnames)}
for er in all_results[:80]:
    r=er['result']
    ax3.scatter(r['w0'],r['wa'],
                color=cmap_b[er['bname']],
                s=40,alpha=0.6,zorder=3)
for bn,col in cmap_b.items():
    ax3.scatter([],[],color=col,s=50,label=bn)
ax3.set_xlabel('w₀'); ax3.set_ylabel('wa')
ax3.set_title('w₀–wa Coverage by β Form')
ax3.set_xlim(-1.2,-0.3); ax3.set_ylim(-1.2,0.5)
ax3.legend(fontsize=7,ncol=2); ax3.grid(True,alpha=0.3)

# Panel 4: Npkt vs wa
ax4=axes[1,0]
npkts=[er['result']['N_pkt'] for er in all_results]
was  =[er['result']['wa']    for er in all_results]
dists=[er['dist']            for er in all_results]
sc=ax4.scatter(npkts,was,c=dists,
               cmap='RdYlGn_r',vmin=0,vmax=0.8,
               s=50,alpha=0.8)
ax4.axhline(wa_t,color='red',ls='--',lw=2,
            label=f'DESI wa={wa_t}')
plt.colorbar(sc,ax=ax4,label='dist to DESI')
ax4.set_xlabel('N_pkt'); ax4.set_ylabel('wa_CPL')
ax4.set_title('wa vs Packet Count (all β forms)')
ax4.legend(fontsize=8); ax4.grid(True,alpha=0.3)

# Panel 5: E_avail(a) for best solutions
ax5=axes[1,1]
for i,er in enumerate(all_results[:4]):
    r=er['result']
    mask=r['a']>=0.1
    ax5.plot(r['a'][mask],
             r['Eav'][mask]/max(r['E_init'],1e-30),
             color=C[i%6],lw=1.8,
             label=(f"{er['bname'][:5]} "
                    f"Npkt={r['N_pkt']}"))
    for ev in r['events']:
        Ne,_,_ = ev
        ax5.axvline(np.exp(Ne),color=C[i%6],
                    ls=':',lw=0.8,alpha=0.5)
ax5.set_xlabel('a')
ax5.set_ylabel('E_avail / E_init')
ax5.set_title('Twist Budget Drain (dashed=event)')
ax5.set_xlim(0.1,1.05)
ax5.legend(fontsize=8); ax5.grid(True,alpha=0.3)

# Panel 6: dist improvement by β form
ax6=axes[1,2]
for bname in bnames:
    sub=[er for er in all_results
         if er['bname']==bname]
    if sub:
        best_d=min(er['dist'] for er in sub)
        med_d =np.median([er['dist'] for er in sub])
        ax6.barh(bname,med_d,
                 xerr=[[med_d-best_d],[0]],
                 color=cmap_b[bname],
                 alpha=0.7, height=0.5,
                 error_kw=dict(lw=2,capsize=4))
ax6.axvline(0.09,color='red',ls='--',lw=1.5,
            label='~1σ threshold')
ax6.set_xlabel('Median dist to DESI')
ax6.set_title('Distance by β Form\n(bar=median, whisker=best)')
ax6.legend(fontsize=8); ax6.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('tafa_phase5_beta.png',dpi=150,
            bbox_inches='tight')
print("\nSaved: tafa_phase5_beta.png")

# ════════════════════════════════════════════════════════
# VERDICT
# ════════════════════════════════════════════════════════
print()
print("="*60)
print("VERDICT")
print("="*60)
if all_results:
    best=all_results[0]
    r=best['result']
    print(f"\nBest solution:")
    print(f"  β-form  : {best['bname']}")
    print(f"  params  : {best['bp']}")
    print(f"  f={best['cfg']['f']}  "
          f"Λ⁴={best['cfg']['Lam4']:.3f}  "
          f"α={best['cfg']['alpha']:.3f}")
    print(f"  ε★_frac={best['ef']:.2f}  "
          f"γ={best['gam']:.2f}  "
          f"band_lo={best['rlof']:.2f}")
    print(f"  Npkt={r['N_pkt']}  "
          f"E_init={r['E_init']:.5f}  "
          f"ε★={r['eps_star']:.5f}")
    print(f"  w_today={r['w_today']:.4f}  "
          f"w₀={r['w0']:.4f}  "
          f"wa={r['wa']:.4f}  "
          f"Ω_φ={r['Op']:.4f}")
    print(f"  dist = {best['dist']:.4f}")
    print()

    # Check which β forms gave Npkt > 0
    pkt_forms={er['bname']
               for er in all_results
               if er['result']['N_pkt']>0}
    print(f"β forms with Npkt>0: "
          f"{pkt_forms if pkt_forms else 'none'}")

    # Best wa achieved
    best_wa=min(all_results,key=lambda x:
                abs(x['result']['wa']-wa_t))
    print(f"\nClosest wa to target: "
          f"wa={best_wa['result']['wa']:.4f} "
          f"(target {wa_t:.3f})  "
          f"β={best_wa['bname']}")

    if best['dist']<0.10:
        print("\n→ WITHIN ~1σ DESI contour.")
        print("  β(ρ) gating is the key mechanism.")
        print("  Verify Npkt events are physically spaced.")
    elif best['dist']<0.20:
        print("\n→ Meaningful improvement over n=0 baseline.")
        print("  Refine β parameters and band geometry.")
        print("  Focus on forms with Npkt > 0.")
    else:
        print("\n→ β(ρ) gating alone is not sufficient.")
        print("  The wa gap remains the binding constraint.")
        print()
        print("Remaining physical options:")
        print("  1. The DESI CPL fit reflects data tension,")
        print("     not necessarily a single rolling field.")
        print("  2. Phantom crossing (w < -1) is required,")
        print("     ruled out by canonical scalar field.")
        print("  3. A non-canonical kinetic term could")
        print("     unlock the needed trajectory shape.")
        print("  4. Two-field models (quintom) cross w=-1.")
        print("  5. This model class has a hard boundary")
        print("     at wa ~ -0.3 for canonical quintessence.")
else:
    print("No valid solutions found.")
    print("Check that n=1 configs survive Ω_φ filter.")

TAFA — PHASE 5: GEOMETRIC TWIST GATING
Non-constant β(ρ) with n=1 controlled release

──────────────────────────────────────────────────────────────────────
Scan: β(ρ) form × (f,Λ,α) × ε★ × γ × band
──────────────────────────────────────────────────────────────────────
    β-form    f  n   ε_f     γ  Npkt  w_today       w₀       wa    Ω_φ   dist
───────────────────────────────────────────────────────────────────────────
    powerl  0.3  1  0.05  0.70     0  -0.6686   0.4424   0.8873 0.7119 2.0718
    powerl  0.3  1  0.05  0.70     0  -0.6686   0.4424   0.8873 0.7119 2.0718
    powerl  0.3  1  0.05  0.70     0  -0.6686   0.4424   0.8873 0.7119 2.0718
    powerl  0.3  1  0.05  0.80     0  -0.6686   0.4424   0.8873 0.7119 2.0718
    powerl  0.3  1  0.05  0.80     0  -0.6686   0.4424   0.8873 0.7119 2.0718
    powerl  0.3  1  0.05  0.80     0  -0.6686   0.4424   0.8873 0.7119 2.0718
    powerl  0.3  1  0.05  0.90     0  -0.6686   0.4424   0.8873 0.7119 2.0718
    powerl  0.3  1  0.05  0.90